In [1]:
import time, os, gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# CUBLAS_WORKSPACE_CONFIG НЕ включаем: окружение новых сидов должно
# совпадать с завершёнными прогонами (смешивать числовые среды между
# сидами таблицы не хотим); детерминизм уже показан эмпирически
# (каузальный тест, §5.5: бит-в-бит воспроизведение)

In [2]:
# ============================================================
# CELL 1: Imports
# ============================================================
import torch
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    get_cosine_schedule_with_warmup,
    set_seed,
)
from peft import PrefixTuningConfig, TaskType, get_peft_model
from datasets import load_dataset

import numpy as np
import evaluate

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {vram_gb:.1f} GB")


<VENV>/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
VRAM: 15.5 GB


In [3]:
# ============================================================
# CELL 2: Config — исследование-2, итерация D1.6
# Апгрейд по итогам внешней критики:
#   1) модель 1.5B -> 3B (реально крупнее статьи);
#   2) настоящий батч 8x1 вместо 1x8 (последовательности ~100 токенов);
#   3) ЧИСТЫЙ eval: вход больше НЕ содержит золотую метку (баг D1.x:
#      generate() продолжал ЗОЛОТУЮ метку — «предсказание» было эхом);
#   4) вербализатор-скоринг (протокол статьи: top-scoring output token);
#   5) per-example корректности -> парный бутстреп (стат. мощность);
#   6) жёсткие негативы для wrong_label (путаемые группы эмоций);
#   7) значения статьи β=0.3, τ=0.1 отдельным конфигом.
# ============================================================
class CFG:
    # ---- 1. Core
    seed = 42
    output_dir = "./gen_dialogue_ed_full"

    gradient_checkpointing = False
    use_cache = True
    bf16 = True
    fp16 = False

    # ---- 2. Model / PEFT
    # D1.6: Qwen2.5-3B (~6.2 ГБ bf16) — заметно крупнее Falcon-rw-1b
    # из статьи (критика: 1.5B был тем же классом). Влезает в 16 ГБ
    # с батчем 8 при seq ~110.
    model_name = "Qwen/Qwen2.5-3B"
    trust_remote_code = True
    peft_kind = "prefix_tuning"
    peft_method = "prefix_tuning"
    num_virtual_tokens = 20
    prefix_projection = True

    # Probe = ГЕНЕРАТОР общего старта
    init_selection_inits = 4
    init_selection_probe_steps = 200

    # ---- 3. Task / Dataset
    task_kind = "emotion_cls"
    dataset_repo = "empathetic_dialogues"
    dataset_revision = "refs/convert/parquet"
    dataset_name = None  # легаси XSum удалено (критика-2: техдолг);
                         # ветка не-cls сейчас не используется

    text_column = "text"
    target_column = "label"

    train_size = 6000
    val_size = 200
    test_size = 500

    max_source_len = 96
    max_target_len = 8
    max_total_len = max_source_len + max_target_len

    prompt_template = "Situation: {source}\nEmotion:"
    padding_side = "left"

    eval_max_new_tokens = 6
    gen_max_new_tokens = 6

    if task_kind == "emotion_cls":
        eval_metrics = ["accuracy", "verb_accuracy"]
    else:
        eval_metrics = ["rouge1", "rouge2", "rougeL"]

    # ---- 4. Schedule
    # D1.6: настоящий батч 8 x accum 1 (та же эффективная партия 8,
    # но GPU-утилизация вместо цикла микробатчей — прогоны в разы
    # быстрее; статистика градиента та же, аккумулирование == батчингу).
    # Эпох 4: 3B сходится быстрее, best-эпохи в D1.5 были 1-3.
    phase1_epochs = 4
    total_epochs = 4
    batch_size = 8
    eval_batch_size = 8
    gradient_accumulation_steps = 1

    learning_rate = 5e-4
    phase2_lr = 1e-4
    weight_decay = 0.0
    warmup_steps = 200

    lr_scheduler = "cosine"
    max_grad_norm = 1.0
    logging_steps = 10
    eval_strategy = "epoch"
    save_strategy = "epoch"
    save_total_limit = 1
    load_best_model_at_end = False
    remove_unused_columns = False
    dataloader_pin_memory = True
    report_to = "none"

    # ---- 5. Reward
    gen_do_sample = False
    gen_temperature = 0.7
    gen_top_p = 0.9
    reward_metric = "accuracy"

    # ---- 6. Contrast
    alpha = 1.0
    beta = 0.1
    contrast_mode = "mask"
    contrast_from_start = True
    contrastive_dropout = 0.1
    contrastive_tau = 0.5
    k_negatives = 1

    gamma = 2e-5
    gamma_auto = True
    gamma_target_frac = 0.3
    sigma = 0.02
    rl_interval = 50
    rl_num_batches = 16
    rl_subset_size = max(16, int(train_size * 0.1))
    use_reward_baseline = True

    divergence_ce_threshold = 8.0
    guard_start_step = 100

    # D1.6: отбор лучшей эпохи по ВСЕМ 200 val-примерам (было 64)
    val_rouge_examples = 200


cfg = CFG()

assert cfg.peft_method == "prefix_tuning"
assert cfg.gradient_checkpointing is False

print("Config OK (D1.6)")
print(f"Model: {cfg.model_name}  <-- апгрейд 1.5B -> 3B")
print(f"Task: {cfg.task_kind} | metrics: {cfg.eval_metrics}")
print(f"PEFT: {cfg.peft_kind} (prefix_projection={cfg.prefix_projection})")
print(f"Dataset: {cfg.dataset_repo} @ {cfg.dataset_revision}")
print(f"Train {cfg.train_size} | val {cfg.val_size} (полная, отбор эпохи) | test {cfg.test_size}")
print(f"Batch: {cfg.batch_size} x accum {cfg.gradient_accumulation_steps} (настоящий батч)")
print(f"Epochs: {cfg.total_epochs} (одна фаза, контраст с 1-й эпохи)")
print(f"LR: {cfg.learning_rate}, warmup: {cfg.warmup_steps}")
print(f"Contrast: mode={cfg.contrast_mode}, beta={cfg.beta}, tau={cfg.contrastive_tau}")
print(f"Eval: ЧИСТЫЙ (без золотой метки во входе) + вербализатор")


Config OK (D1.6)
Model: Qwen/Qwen2.5-3B  <-- апгрейд 1.5B -> 3B
Task: emotion_cls | metrics: ['accuracy', 'verb_accuracy']
PEFT: prefix_tuning (prefix_projection=True)
Dataset: empathetic_dialogues @ refs/convert/parquet
Train 6000 | val 200 (полная, отбор эпохи) | test 500
Batch: 8 x accum 1 (настоящий батч)
Epochs: 4 (одна фаза, контраст с 1-й эпохи)
LR: 0.0005, warmup: 200
Contrast: mode=mask, beta=0.1, tau=0.5
Eval: ЧИСТЫЙ (без золотой метки во входе) + вербализатор


In [4]:
# ============================================================
# CELL 3: Data loading & tokenization
# D1.6: жёсткие негативы wrong_label из групп путаемых эмоций
# (критика D1.5: случайная чужая метка слишком легка) + первые
# токены меток для вербализатор-скоринга
# ============================================================
import re
import random
from huggingface_hub import hf_hub_download, list_repo_files
from datasets import Dataset as HFDataset

tok = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
tok.padding_side = cfg.padding_side

if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id

if cfg.task_kind == "emotion_cls":
    def load_ed_episodes(split):
        """ED — script-датасет, новыми datasets не грузится; читаем
        parquet-конвертацию с Hub. Эпизоды = уникальные conv_id."""
        files = list_repo_files(cfg.dataset_repo, repo_type="dataset",
                                revision=cfg.dataset_revision)
        names = sorted(f for f in files
                       if f.startswith(f"default/{split}/") and f.endswith(".parquet"))
        assert names, f"нет parquet-шардов для split={split}"
        paths = [hf_hub_download(cfg.dataset_repo, n, repo_type="dataset",
                                 revision=cfg.dataset_revision) for n in names]
        ds = load_dataset("parquet", data_files=paths, split="train")
        episodes, seen = [], set()
        for r in ds:
            cid = r["conv_id"]
            if cid in seen:
                continue
            seen.add(cid)
            text, label = r["prompt"].strip(), r["context"].strip()
            if text and label:
                episodes.append({"text": text, "label": label})
        return episodes

    ed_train_full = load_ed_episodes("train")
    ed_val_full = load_ed_episodes("validation")
    ed_test_full = load_ed_episodes("test")
    print(f"ED episodes: train {len(ed_train_full)} | "
          f"val {len(ed_val_full)} | test {len(ed_test_full)}")

    emotion_labels = sorted({e["label"] for e in ed_train_full})
    assert len(emotion_labels) == 32, f"ожидалось 32 эмоции, есть {len(emotion_labels)}"

    def _norm(s):
        return re.sub(r"[^a-z ]", "", s.lower()).strip()

    EMOTION_BY_NORM = {_norm(l): l for l in emotion_labels}

    def match_emotion(text):
        g = _norm(text)
        if not g:
            return None
        first = g.split()[0]
        if first in EMOTION_BY_NORM:
            return EMOTION_BY_NORM[first]
        if g in EMOTION_BY_NORM:
            return EMOTION_BY_NORM[g]
        for nl, l in EMOTION_BY_NORM.items():
            if g.startswith(nl):
                return l
        for nl, l in EMOTION_BY_NORM.items():
            if nl in g:
                return l
        return None

    LABEL_TOK_IDS = {
        lab: tok(" " + lab, add_special_tokens=False,
                 truncation=True, max_length=cfg.max_target_len)["input_ids"]
        for lab in emotion_labels
    }

    # D1.6: первые токены меток — для вербализатор-скоринга
    # (протокол статьи: top-scoring output token). Проверяем
    # единственность: две метки с одним первым токеном неразличимы.
    LABEL_FIRST_TOKEN = {lab: ids[0] for lab, ids in LABEL_TOK_IDS.items()}
    _firsts = list(LABEL_FIRST_TOKEN.values())
    assert len(set(_firsts)) == len(_firsts), (
        "коллизия первых токенов меток — вербализатор неразличим: " +
        str([l for l in LABEL_FIRST_TOKEN
             if list(LABEL_FIRST_TOKEN.values()).count(LABEL_FIRST_TOKEN[l]) > 1])
    )
    _multi = [l for l in emotion_labels if len(LABEL_TOK_IDS[l]) > 1]
    print(f"Вербализатор: все 32 первых токена уникальны; "
          f"мультитокенных меток: {len(_multi)} {_multi}")

    # D1.6: ЖЁСТКИЕ негативы — группы путаемых эмоций (критика D1.5:
    # случайная чужая метка из 31 слишком легка; негатив должен быть
    # реальной конкурирующей альтернативой)
    CONFUSION_GROUPS = [
        {"afraid", "terrified", "anxious", "apprehensive"},
        {"sad", "disappointed", "devastated", "lonely"},
        {"angry", "furious", "annoyed", "disgusted"},
        {"surprised", "anticipating", "excited", "joyful"},
        {"guilty", "ashamed", "embarrassed"},
        {"hopeful", "faithful", "trusting", "prepared", "confident", "proud"},
        {"caring", "grateful", "impressed"},
        {"sentimental", "nostalgic", "content"},
        {"jealous"},
    ]
    LABEL_GROUP = {}
    for g in CONFUSION_GROUPS:
        for lab in g:
            if lab in emotion_labels:
                LABEL_GROUP[lab] = g
    print(f"Группы путаемости покрывают {len(LABEL_GROUP)}/32 меток")

    _neg_rng = random.Random(1234)

    def sample_wrong_label(true_label):
        """Жёсткий негатив: предпочтительно из группы путаемости."""
        group = LABEL_GROUP.get(true_label)
        if group and len(group) > 1:
            pool = [l for l in group if l != true_label]
            return _neg_rng.choice(pool)
        pool = [l for l in emotion_labels if l != true_label]
        return _neg_rng.choice(pool)

    raw_rl    = HFDataset.from_list(ed_train_full[:cfg.rl_subset_size])
    raw_train = HFDataset.from_list(ed_train_full[cfg.rl_subset_size:
                                                  cfg.rl_subset_size + cfg.train_size])
    raw_val   = HFDataset.from_list(ed_val_full[:cfg.val_size])
    raw_test  = HFDataset.from_list(ed_test_full[:cfg.test_size])

    assert cfg.rl_subset_size + cfg.train_size <= len(ed_train_full), \
        "hold-out + train не помещаются в датасет"

    print(f"Эмоции (32): {emotion_labels}")
else:
    raw = load_dataset(cfg.dataset_name)
    print("Splits:", list(raw.keys()))
    raw_rl    = raw["train"].select(range(0, cfg.rl_subset_size))
    raw_train = raw["train"].select(range(cfg.rl_subset_size,
                                          cfg.rl_subset_size + cfg.train_size))
    raw_val   = raw["validation"].select(range(cfg.val_size))
    raw_test  = raw["test"].select(range(cfg.test_size))
    assert cfg.rl_subset_size + cfg.train_size <= len(raw["train"])

def tokenize_gen(examples):
    all_input_ids = []
    all_attention_mask = []
    all_labels = []
    all_neg_input_ids = []
    all_neg_attention_mask = []
    all_neg_labels = []

    max_total = cfg.max_total_len
    tgt_prefix = " " if cfg.task_kind == "emotion_cls" else ""
    add_eos = cfg.task_kind != "emotion_cls"  # D1.1b: без EOS в cls

    for s, t in zip(examples[cfg.text_column], examples[cfg.target_column]):
        src = cfg.prompt_template.format(source=s)

        src_ids = tok(
            src,
            add_special_tokens=False,
            truncation=True,
            max_length=cfg.max_source_len,
        )["input_ids"]

        tgt_max = max(1, cfg.max_target_len - 1) if add_eos else cfg.max_target_len
        tgt_ids = tok(
            tgt_prefix + t,
            add_special_tokens=False,
            truncation=True,
            max_length=tgt_max,
        )["input_ids"]
        if add_eos:
            tgt_ids = tgt_ids + [tok.eos_token_id]

        if len(src_ids) == 0:
            src_ids = [tok.pad_token_id]

        if len(src_ids) + len(tgt_ids) > max_total:
            src_ids = src_ids[:max(1, max_total - len(tgt_ids))]

        input_ids = src_ids + tgt_ids
        labels = [-100] * len(src_ids) + tgt_ids

        all_input_ids.append(input_ids)
        all_attention_mask.append([1] * len(input_ids))
        all_labels.append(labels)

        # D1.6: ЖЁСТКИЙ негатив wrong-label — (ситуация, путаемая эмоция)
        if cfg.task_kind == "emotion_cls":
            wrong_ids = list(LABEL_TOK_IDS[sample_wrong_label(t)])
            neg_ids = src_ids + wrong_ids
            all_neg_input_ids.append(neg_ids)
            all_neg_attention_mask.append([1] * len(neg_ids))
            all_neg_labels.append([-100] * len(src_ids) + wrong_ids)

    out = {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    }
    if cfg.task_kind == "emotion_cls":
        out.update({
            "neg_input_ids": all_neg_input_ids,
            "neg_attention_mask": all_neg_attention_mask,
            "neg_labels": all_neg_labels,
        })
    return out

train_ds = raw_train.map(tokenize_gen, batched=True, remove_columns=raw_train.column_names)
val_ds = raw_val.map(tokenize_gen, batched=True, remove_columns=raw_val.column_names)
test_ds = raw_test.map(tokenize_gen, batched=True, remove_columns=raw_test.column_names)
rl_ds = raw_rl.map(tokenize_gen, batched=True, remove_columns=raw_rl.column_names)

rl_ds = rl_ds.add_column("ref_idx", list(range(len(rl_ds))))

rl_references = raw_rl[cfg.target_column]
val_references = raw_val[cfg.target_column]
test_references = raw_test[cfg.target_column]

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")
rl_ds.set_format("torch")

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} | "
      f"RL hold-out: {len(rl_ds)} (не входит в train)")
assert (train_ds[0]["labels"] != -100).sum() > 0, "нет супервизируемых токенов"
if cfg.task_kind == "emotion_cls":
    print("Sample NEG (последние 10 токенов):", train_ds[0]["neg_labels"][-10:])


ED episodes: train 17844 | val 2763 | test 2542
Вербализатор: все 32 первых токена уникальны; мультитокенных меток: 1 ['apprehensive']
Группы путаемости покрывают 32/32 меток
Эмоции (32): ['afraid', 'angry', 'annoyed', 'anticipating', 'anxious', 'apprehensive', 'ashamed', 'caring', 'confident', 'content', 'devastated', 'disappointed', 'disgusted', 'embarrassed', 'excited', 'faithful', 'furious', 'grateful', 'guilty', 'hopeful', 'impressed', 'jealous', 'joyful', 'lonely', 'nostalgic', 'prepared', 'proud', 'sad', 'sentimental', 'surprised', 'terrified', 'trusting']


Map: 100%|██████████| 600/600 [00:00<00:00, 8617.19 examples/s]

Train: 6000 | Val: 200 | Test: 500 | RL hold-out: 600 (не входит в train)
Sample NEG (последние 10 токенов): tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 36413])


In [5]:
def label_stats(ds, n=100):
    n = min(n, len(ds))
    examples_with_target = 0
    total_target_tokens = 0

    for i in range(n):
        labs = ds[i]["labels"]
        if isinstance(labs, torch.Tensor):
            labs = labs.tolist()

        cnt = sum(1 for x in labs if x != -100)
        total_target_tokens += cnt
        if cnt > 0:
            examples_with_target += 1

    print(f"checked: {n}")
    print(f"examples_with_target: {examples_with_target}")
    print(f"avg target tokens: {total_target_tokens / n:.2f}")

label_stats(train_ds)
label_stats(val_ds)

checked: 100
examples_with_target: 100
avg target tokens: 1.03
checked: 100
examples_with_target: 100
avg target tokens: 1.05


In [6]:
# ============================================================
# CELL 4: Model + PEFT (Prefix Tuning for Causal LM)
# ============================================================
# Детерминированная инициализация префикса (важно для честного
# сравнения baseline vs full: seed фиксируется ДО get_peft_model)
set_seed(cfg.seed)

dtype = torch.bfloat16 if cfg.bf16 else torch.float32

base = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    dtype=dtype,
    trust_remote_code=True,
)

peft_cfg = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=cfg.num_virtual_tokens,
    prefix_projection=cfg.prefix_projection,
    inference_mode=False,
)

model = get_peft_model(base, peft_cfg)

# Обучаемые параметры (prompt encoder) в fp32: bf16-веса + AdamW —
# известный источник нестабильности; при скачке лосса Tanh в MLP
# насыщается, градиенты умирают, и модель застревает навсегда
# (baseline прошлого прогона завис на CE~6.5 на 5.5 эпох).
# Автокаст bf16 в Trainer корректно работает со смешанными dtype.
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

model.enable_input_require_grads()

model.print_trainable_parameters()

prompt_params = [
    (n, (tuple(p.shape), p.dtype))
    for n, p in model.named_parameters()
    if "prompt_encoder" in n
]

print(f"Prompt parameters found: {len(prompt_params)}")
for n, (s, d) in prompt_params[:6]:
    print(f"  {n}: {s} ({d})")


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10156.04it/s]


trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
Prompt parameters found: 5
  prompt_encoder.default.embedding.weight: (20, 256) (torch.float32)
  prompt_encoder.default.transform.0.weight: (256, 256) (torch.float32)
  prompt_encoder.default.transform.0.bias: (256,) (torch.float32)
  prompt_encoder.default.transform.2.weight: (18432, 256) (torch.float32)
  prompt_encoder.default.transform.2.bias: (18432,) (torch.float32)


In [7]:
# ============================================================
# CELL 5: Reward function
# D1 (emotion_cls): accuracy матчинга сгенерированного слова к 32 меткам
# summarization (наследие): ROUGE-L против референса
# ============================================================
rouge_metric = evaluate.load("rouge")

@torch.no_grad()
def compute_generation_reward(model, input_ids, attention_mask, references, tokenizer,
                              task_cfg=None):
    """Reward на ДАННОМ примере против его референса. Greedy ->
    детерминированный reward (аналог accuracy в статье).

    Для task_kind="emotion_cls": references = строки-эмоции,
    reward = доля примеров, где match_emotion(генерация) == метка.
    Для "summarization": reward = ROUGE-L.
    """
    # D2.2: task_cfg — конфиг ЗАДАЧИ (глобальный cfg может быть чужим,
    # например D1-классификацией); по умолчанию прежнее поведение
    c = task_cfg if task_cfg is not None else cfg
    model.eval()

    old_use_cache = getattr(model.config, "use_cache", True)
    model.config.use_cache = True

    try:
        gen_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=c.gen_max_new_tokens,
            do_sample=c.gen_do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if c.gen_do_sample:
            gen_kwargs.update(temperature=c.gen_temperature, top_p=c.gen_top_p)

        gen_ids = model.generate(**gen_kwargs)
    finally:
        model.config.use_cache = old_use_cache

    new_tokens = gen_ids[:, input_ids.shape[1]:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    generated_texts = [x.strip() if x.strip() else " " for x in generated_texts]

    assert len(generated_texts) == len(references), \
        f"predictions/references mismatch: {len(generated_texts)} vs {len(references)}"

    if c.task_kind == "emotion_cls":
        reward = float(np.mean([
            match_emotion(g) == r
            for g, r in zip(generated_texts, references)
        ]))
    else:
        results = rouge_metric.compute(
            predictions=generated_texts,
            references=references,
            rouge_types=["rougeL"],
        )
        reward = float(results["rougeL"])

    model.train()
    return reward, generated_texts

print("Reward function defined: accuracy (emotion_cls) / ROUGE-L (summarization).")


Reward function defined: accuracy (emotion_cls) / ROUGE-L (summarization).


In [8]:
# ============================================================
# CELL 6: TwoPhaseTrainerGen — адаптация Algorithm 1
# D1.5: contrast_mode = "mask" | "wrong_label"; contrast_from_start
# D1.6: RL reward на ЧИСТОМ входе (strip золотой метки — баг эха)
# ============================================================
class TwoPhaseTrainerGen(Trainer):
    def __init__(
        self,
        model=None,
        args=None,
        phase1_epochs: int = 3,
        alpha: float = 1.0,
        beta: float = 0.1,
        gamma: float = 2e-5,
        gamma_auto: bool = True,
        gamma_target_frac: float = 0.3,
        sigma: float = 0.02,
        contrast_mode: str = "mask",
        contrast_from_start: bool = False,
        contrastive_dropout: float = 0.1,
        contrastive_tau: float = 0.5,
        k_negatives: int = 1,
        rl_interval: int = 50,
        rl_num_batches: int = 16,
        rl_subset_size: int = 200,
        use_reward_baseline: bool = True,
        phase2_lr: float = 2e-4,
        divergence_ce_threshold: float = 6.0,
        guard_start_step: int = 200,
        rl_dataset=None,
        rl_references=None,
        tokenizer=None,
        task_cfg=None,
        contrast_neg_in_graph: bool = False,
        reward_baseline_beta: float = 0.9,
        **kwargs
    ):
        self.phase1_epochs = phase1_epochs
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.gamma_auto = gamma_auto
        self.gamma_target_frac = gamma_target_frac
        self.sigma = sigma
        self.contrast_mode = contrast_mode
        self.contrast_from_start = contrast_from_start
        self.contrastive_dropout = contrastive_dropout
        self.contrastive_tau = contrastive_tau
        self.k_negatives = k_negatives
        self.rl_interval = rl_interval
        self.rl_num_batches = rl_num_batches
        self.rl_subset_size = rl_subset_size
        self.use_reward_baseline = use_reward_baseline
        self.phase2_lr = phase2_lr
        self.divergence_ce_threshold = divergence_ce_threshold
        self.guard_start_step = guard_start_step
        self.rl_dataset = rl_dataset
        self.rl_references = rl_references
        self.tokenizer = tokenizer
        self.task_cfg = task_cfg          # D2.2: конфиг задачи для reward
        self.rl_reward_history = []       # D2.2: трекинг наград по шагам
        self.contrast_neg_in_graph = contrast_neg_in_graph  # D2.3
        self.reward_baseline_beta = reward_baseline_beta    # D2.2b

        # Флаги состояний
        self._phase2_lr_reset_done = False
        self._gamma_calibrated = False
        self._last_rl_step = -1
        self._last_log_step = -1
        self._reward_baseline = 0.0
        self._ce_ema = None
        self._divergence_reported = False

        super().__init__(model=model, args=args, **kwargs)

        # Параметры префикса собираем ПОСЛЕ super().__init__
        self.prefix_params = [
            (n, p) for n, p in self.model.named_parameters()
            if "prompt_encoder" in n and p.requires_grad
        ]
        print(f"[Trainer] Prefix params collected: {len(self.prefix_params)}")

        # Цели контраста (для mode="mask") — ТОЛЬКО embedding-слой
        self.contrast_params = [
            (n, p) for n, p in self.prefix_params if "embedding" in n
        ] or self.prefix_params
        print(f"[Trainer] Contrast: mode={self.contrast_mode}, "
              f"from_start={self.contrast_from_start}, beta={self.beta}")

        if self.rl_dataset is not None:
            self._rl_loader = DataLoader(
                self.rl_dataset,
                batch_size=self.args.per_device_train_batch_size,
                collate_fn=self.data_collator,
            )
            self._rl_iter = iter(self._rl_loader)

    # ----------------------------------------------------------
    # Утилиты
    # ----------------------------------------------------------
    @staticmethod
    def _model_inputs(inputs):
        """Только те ключи, которые понимает model.forward. Коллатор
        дополнительно кладёт neg_*/ref_idx — их в модель нельзя."""
        return {k: v for k, v in inputs.items()
                if k in ("input_ids", "attention_mask", "labels")}

    def _next_rl_batch(self):
        try:
            return next(self._rl_iter)
        except StopIteration:
            self._rl_iter = iter(self._rl_loader)
            return next(self._rl_iter)

    def _restore_params(self, params, snapshot):
        with torch.no_grad():
            for n, p in params:
                p.data.copy_(snapshot[n])

    def _custom_scale(self):
        """ЕДИНЫЙ множитель для ВСЕГО лосса, включая CE (урок ит. 2
        исследования-1: иначе эффективный LR x gradient_accumulation)."""
        trainer_divides = not getattr(self, "model_accepts_loss_kwargs", False)
        return 1.0 if trainer_divides else 1.0 / self.args.gradient_accumulation_steps

    def _reset_lr_for_phase2(self):
        if self.optimizer is None or self.lr_scheduler is None:
            return
        remaining = max(1, self.state.max_steps - self.state.global_step)
        for pg in self.optimizer.param_groups:
            pg["lr"] = self.phase2_lr
            pg["initial_lr"] = self.phase2_lr
        self.lr_scheduler = get_cosine_schedule_with_warmup(
            self.optimizer, num_warmup_steps=0, num_training_steps=remaining,
        )
        print(f"\n[Phase 2] LR reset to {self.phase2_lr}, "
              f"new cosine over remaining {remaining} steps\n")

    def _pool_hidden(self, hidden, labels):
        """Пуллинг по токенам таргета (labels != -100): для позитива —
        токены верной метки, для негатива — токены чужой метки."""
        if hidden.size(1) != labels.size(1):
            hidden = hidden[:, -labels.size(1):, :]
        tgt_mask = (labels != -100).unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * tgt_mask).sum(1) / tgt_mask.sum(1).clamp(min=1)
        return F.normalize(pooled.float(), dim=-1)

    # ----------------------------------------------------------
    # Contrastive loss — два режима негативов
    # ----------------------------------------------------------
    def _contrastive_loss(self, model, inputs, h_pos):
        """L = log1 + sum_j exp((sim(h_i, h_j^-) - 1) / tau))

        mode="mask" (рецепт статьи): h_j^- — представление ТОГО ЖЕ входа
        под замаскированным embedding-слоем промпта. Позитив тривиален
        (sim(h,h)=1 — вырожденность Eq. 3, см. FINAL_ANALYSIS §4).

        mode="wrong_label" (ядро темы): негатив — (ситуация, ЖЁСТКАЯ
        чужая эмоция из группы путаемости) — реальная конкурирующая
        альтернатива."""
        if self.k_negatives <= 0:
            return h_pos.new_zeros(())

        h_negs = []

        # D2: негативы из neg-колонок, собранных при токенизации
        # (neg = wrong-label для cls; negoff*/neginc для response_gen).
        # D2.3: offtopic4 = k=4 негативов (Σ_j как в Eq.3 статьи);
        # oftopic_grad = градиент и через негатив (симметричный апдейт)
        COLUMN_MODES = {
            "wrong_label": ["neg"],
            "offtopic": ["negoff"],
            "offtopic_grad": ["negoff"],
            "offtopic4": ["negoff", "negoff2", "negoff3", "negoff4"],
            "incoherent": ["neginc"],
        }
        if self.contrast_mode in COLUMN_MODES:
            for pfx in COLUMN_MODES[self.contrast_mode]:
                neg_fwd = {
                    "input_ids": inputs[f"{pfx}_input_ids"],
                    "attention_mask": inputs[f"{pfx}_attention_mask"],
                }
                if getattr(self, "contrast_neg_in_graph", False):
                    out_neg = model(**neg_fwd, output_hidden_states=True)
                else:
                    with torch.no_grad():
                        out_neg = model(**neg_fwd, output_hidden_states=True)
                h_negs.append(self._pool_hidden(
                    out_neg.hidden_states[-1], inputs[f"{pfx}_labels"]))
                del out_neg
        else:
            neg_inputs = {k: v for k, v in self._model_inputs(inputs).items()
                          if k != "labels"}
            labels = inputs["labels"]
            original_data = {n: p.data.clone() for n, p in self.contrast_params}
            try:
                for _ in range(self.k_negatives):
                    for n, p in self.contrast_params:
                        mask = (torch.rand_like(p.float()) > self.contrastive_dropout).to(p.dtype)
                        p.data.copy_((original_data[n] * mask).to(p.dtype))
                    with torch.no_grad():
                        out_neg = model(**neg_inputs, output_hidden_states=True)
                    h_neg = self._pool_hidden(out_neg.hidden_states[-1], labels)
                    h_negs.append(h_neg)
                    del out_neg
                    self._restore_params(self.contrast_params, original_data)
            finally:
                self._restore_params(self.contrast_params, original_data)

        h_negs = torch.stack(h_negs, dim=1)  # (B, K, D)

        tau = self.contrastive_tau
        neg_scores = torch.einsum("bd,bkd->bk", h_pos, h_negs) / tau
        loss = torch.log1p(torch.exp(neg_scores - 1.0 / tau).sum(dim=1)).mean()
        return loss

    # ----------------------------------------------------------
    # RL loss (REINFORCE, Eq. 4 статьи)
    # ----------------------------------------------------------
    def _rl_loss(self, model):
        log_probs = []
        noisy_values = {}
        for n, p in self.prefix_params:
            p32 = p.float()
            eps = torch.randn_like(p32) * self.sigma
            noisy = (p32.detach() + eps).detach()
            log_probs.append(
                Normal(loc=p32, scale=self.sigma).log_prob(noisy).sum()
            )
            noisy_values[n] = noisy

        original_data = {n: p.data.clone() for n, p in self.prefix_params}
        rewards = []
        last_texts, last_refs = None, None
        try:
            for n, p in self.prefix_params:
                p.data.copy_(noisy_values[n].to(p.dtype))

            for _ in range(self.rl_num_batches):
                rl_batch = self._next_rl_batch()
                idx = rl_batch.pop("ref_idx", None)
                if idx is None:
                    raise RuntimeError(
                        "ref_idx потерян коллатором: causal_lm_collator обязан "
                        "пробрасывать ref_idx"
                    )
                # D1.6: ЧИСТЫЙ reward — отрезаем золотую метку до generate
                # (баг эха: иначе reward мерил повторение подсказанной метки)
                src_ids, src_mask = strip_label_tokens(rl_batch)
                batch = {
                    "input_ids": src_ids.to(model.device),
                    "attention_mask": src_mask.to(model.device),
                }
                refs = [self.rl_references[i] for i in idx.tolist()]
                r, texts = compute_generation_reward(
                    model, batch["input_ids"], batch["attention_mask"],
                    refs, self.tokenizer, task_cfg=self.task_cfg,
                )
                rewards.append(r)
                last_texts, last_refs = texts, refs
            reward = float(np.mean(rewards))
            self.rl_reward_history.append(
                {"step": int(self.state.global_step), "reward": reward})
        finally:
            self._restore_params(self.prefix_params, original_data)

        if self.use_reward_baseline:
            advantage = reward - self._reward_baseline
            _beta_b = self.reward_baseline_beta   # D2.2b: инерция EMA
            self._reward_baseline = (
                _beta_b * self._reward_baseline + (1 - _beta_b) * reward)
        else:
            advantage = reward

        raw_rl = -advantage * torch.stack(log_probs).sum()

        if self.gamma_auto and not self._gamma_calibrated:
            raw_abs = abs(float(raw_rl.detach()))
            if raw_abs < 1e-6 or self._ce_ema is None:
                print("[RL] gamma calibration postponed")
            else:
                accum = self.args.gradient_accumulation_steps
                self.gamma = float(np.clip(
                    self.gamma_target_frac * self._ce_ema * accum / raw_abs,
                    1e-8, 1.0,
                ))
                self._gamma_calibrated = True
                print(f"[RL] gamma auto-calibrated to {self.gamma:.3e}")

        loss_rl = self.gamma * raw_rl

        if last_texts is not None:
            print(f"[RL sample] REF: {last_refs[0][:110]!r}")
            print(f"[RL sample] GEN: {last_texts[0][:110]!r}")
        print(f"[RL step {self.state.global_step}] reward={reward:.4f} "
              f"adv={advantage:+.4f} gamma={self.gamma:.2e}")
        return loss_rl, reward

    # ----------------------------------------------------------
    # Основной лосс
    # ----------------------------------------------------------
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        current_epoch = int(self.state.epoch) if self.state.epoch else 0
        in_phase2 = current_epoch >= self.phase1_epochs

        if in_phase2 and not self._phase2_lr_reset_done:
            self._phase2_lr_reset_done = True
            self._reset_lr_for_phase2()

        use_contrast = self.beta != 0.0 and (in_phase2 or self.contrast_from_start)

        model_inputs = self._model_inputs(inputs)
        outputs = model(**model_inputs, output_hidden_states=use_contrast)
        loss_mle = outputs.loss

        ce_now = loss_mle.item()
        self._ce_ema = ce_now if self._ce_ema is None else 0.95 * self._ce_ema + 0.05 * ce_now

        if (self.state.global_step >= self.guard_start_step
                and self._ce_ema > self.divergence_ce_threshold
                and not self._divergence_reported):
            self._divergence_reported = True
            print(f"\n!!! CE/token EMA = {self._ce_ema:.2f} > "
                  f"{self.divergence_ce_threshold} — расходимость. "
                  f"Останавливаю обучение (шаг {self.state.global_step}).\n")
            self.control.should_training_stop = True

        scale = self._custom_scale()

        loss_contrast = loss_mle.new_zeros(())
        if use_contrast:
            h_pos = self._pool_hidden(outputs.hidden_states[-1],
                                      model_inputs["labels"])
            loss_contrast = self._contrastive_loss(model, inputs, h_pos)

        loss_rl = loss_mle.new_zeros(())
        reward_val = 0.0
        rl_enabled = (self.gamma != 0.0 or self.gamma_auto) and self.rl_dataset is not None
        if (
            rl_enabled
            and self.state.global_step % self.rl_interval == 0
            and self._last_rl_step != self.state.global_step
        ):
            self._last_rl_step = self.state.global_step
            loss_rl, reward_val = self._rl_loss(model)

        if (model.training and self.state.global_step % 10 == 0
                and self._last_log_step != self.state.global_step):
            self._last_log_step = self.state.global_step
            print(f"[Step {self.state.global_step}] CE/token (EMA): {self._ce_ema:.3f} | "
                  f"Contrast[{self.contrast_mode}]: {loss_contrast.item():.3f} | "
                  f"RL: {loss_rl.item():.3f} | Reward: {reward_val:.4f}")

        loss = scale * (self.alpha * loss_mle + self.beta * loss_contrast + loss_rl)
        return (loss, outputs) if return_outputs else loss

print("TwoPhaseTrainerGen defined ✓")


TwoPhaseTrainerGen defined ✓


In [9]:
# ============================================================
# CELL 7: collator + ЧИСТЫЙ eval (фикс бага эха) + коллбэки
#
# БАГ ПРОТОКОЛА D1.x (найден при разборе критики): вход для generate
# СОДЕРЖАЛ золотую метку в конце -> модель продолжала её, и
# «предсказанием» было ЭХО метки. Канарейка D1.5 это показывает
# буквально: «surprised surprised surprised…» — повтор золотой метки.
# Отсюда подозрительно высокие accuracy обученных моделей (0.45–0.65)
# при чистом zero-shot 0.18. Фикс: перед generate отрезаем токены
# метки (labels != -100 считаются от КОНЦА последовательности).
# ============================================================
import json as _json
import torch.nn.functional as F  # ре-импорт: защита от затенения имени F
                                  # (баг: ячейка BERTScore писала P, R, F = ...)


def strip_label_tokens(batch):
    """Отрезает токены метки в конце каждого примера и заново
    лево-пэддит батч. Возвращает (input_ids, attention_mask)
    src-only — модель должна ПРЕДСКАЗАТЬ метку, а не прочитать."""
    labels = batch["labels"]
    n_lab = (labels != -100).sum(dim=1)
    L = batch["input_ids"].size(1)
    srcs = [batch["input_ids"][i, :L - int(n_lab[i])]
            for i in range(batch["input_ids"].size(0))]
    # Маску берём ТЕМ ЖЕ срезом из оригинала: в отрезанном куске
    # остаются исходные лево-паддинговые позиции с маской 0
    # (юнит-тест поймал баг: маска «все единицы» ломала позициями)
    ams = [batch["attention_mask"][i, :L - int(n_lab[i])]
           for i in range(batch["input_ids"].size(0))]
    max_len = max(s.size(0) for s in srcs)
    input_ids = torch.stack([
        s if s.size(0) == max_len
        else F.pad(s, (max_len - s.size(0), 0), value=tok.pad_token_id)
        for s in srcs
    ])
    attention_mask = torch.stack([
        m if m.size(0) == max_len
        else F.pad(m, (max_len - m.size(0), 0), value=0)
        for m in ams
    ])
    return input_ids, attention_mask


@torch.no_grad()
def verbalizer_eval(model, dataset, references, tokenizer, batch_size=16):
    """Вербализатор-скоринг (протокол статьи: top-scoring output token):
    ОДИН forward на src-only входе, argmax по первым токенам 32 меток
    в последней позиции. Детерминированно, без генерации.
    Возвращает (accuracy, correct_vec)."""
    if cfg.task_kind != "emotion_cls":
        return None, None
    model.eval()
    order_labels = sorted(LABEL_FIRST_TOKEN.keys())
    tok_ids = torch.tensor([LABEL_FIRST_TOKEN[l] for l in order_labels],
                           device=model.device)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=causal_lm_collator)
    correct = []
    idx = 0
    for batch in loader:
        input_ids, attention_mask = strip_label_tokens(batch)
        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device)
        logits = model(input_ids=input_ids,
                       attention_mask=attention_mask).logits[:, -1, :]
        pred_idx = logits[:, tok_ids].argmax(dim=-1).tolist()
        for pi in pred_idx:
            correct.append(int(order_labels[pi] == references[idx]))
            idx += 1
    return float(np.mean(correct)), correct


def causal_lm_collator(features):
    pad_token_id = tok.pad_token_id

    def to_1d_tensor(x, dtype=torch.long):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=dtype)
        if x.dim() > 1:
            x = x.squeeze(0)
        return x

    def pad_batch(seqs, value=0):
        max_len = max(s.size(0) for s in seqs)
        return torch.stack([
            s if s.size(0) == max_len
            else F.pad(s, (max_len - s.size(0), 0), value=value)
            for s in seqs
        ])

    # ПАДДИНГ = 0 ДЛЯ ОБОИХ (легаси всех завершённых прогонов).
    # Эмпирически доказано (журнал §6.3): в этом стеке значение пэда
    # input_ids ВЛИЯЕТ на логиты реальных позиций (max diff 6.2 даже
    # при нулевой маске) — «гигиена» pad_token_id НЕ числово-нейтральна,
    # а attention_mask с ненулевыми падами ломает маскирование вовсе.
    # Поэтому: (1) оставляем 0/0 как во всех проведённых экспериментах;
    # (2) замена на pad_token_id возможна ТОЛЬКО при полном перезапуске.
    # Известная историческая особенность: eval-пути (strip_label_tokens,
    # zero-shot через токенизатор) паддят pad_token_id — train/eval
    # рассогласование существовало ВСЕГДА, одинаково для всех конфигов
    # (парные выводы не затронуты), задокументировано в §6.3.
    out = {
        "input_ids": pad_batch([to_1d_tensor(f["input_ids"]) for f in features]),
        "attention_mask": pad_batch(
            [to_1d_tensor(f["attention_mask"]) for f in features]),
    }
    # labels: левый паддинг значением -100
    lab_list = []
    max_len = out["input_ids"].size(1)
    for f in features:
        labs = to_1d_tensor(f["labels"])
        pad_len = max_len - labs.size(0)
        lab_list.append(labs if pad_len == 0 else F.pad(labs, (pad_len, 0), value=-100))
    out["labels"] = torch.stack(lab_list)

    # Негативные последовательности: neg (wrong-label, cls),
    # negoff / negoff2..4 / neginc (D2; D2.3 — динамическое
    # обнаружение любых *_input_ids-колонок негативов)
    neg_pfxs = sorted({k[:-len("_input_ids")] for k in features[0]
                       if k.endswith("_input_ids") and k != "input_ids"})
    for pfx in neg_pfxs:
        if f"{pfx}_attention_mask" in features[0]:
            n_ids = [to_1d_tensor(f[f"{pfx}_input_ids"]) for f in features]
            n_mask = [to_1d_tensor(f[f"{pfx}_attention_mask"]) for f in features]
            n_labs = [to_1d_tensor(f[f"{pfx}_labels"]) for f in features]
            max_len = max(s.size(0) for s in n_ids)
            out[f"{pfx}_input_ids"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=pad_token_id)
                for s in n_ids])
            out[f"{pfx}_attention_mask"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=0)
                for s in n_mask])
            out[f"{pfx}_labels"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=-100)
                for s in n_labs])

    if "ref_idx" in features[0]:
        out["ref_idx"] = torch.tensor(
            [int(f["ref_idx"]) for f in features], dtype=torch.long
        )

    return out


class Phase1CanaryCallback(TrainerCallback):
    """Каждую эпоху печатает ЧИСТЫЕ генерации (src-only) на val;
    если ВСЕ пустые — останавливает обучение."""

    def __init__(self, canary_ds, tokenizer, collator, first_epoch=2,
                 num_examples=4, max_new_tokens=6):
        n = min(num_examples, len(canary_ds))
        self.loader = DataLoader(
            canary_ds.select(range(n)), batch_size=1, collate_fn=collator
        )
        self.tokenizer = tokenizer
        self.first_epoch = first_epoch
        self.max_new_tokens = max_new_tokens
        self.stopped = False

    @torch.no_grad()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None or state.epoch < self.first_epoch or self.stopped:
            return

        was_training = model.training
        model.eval()
        old_use_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True

        texts = []
        try:
            for batch in self.loader:
                input_ids, attention_mask = strip_label_tokens(batch)
                input_ids = input_ids.to(model.device)
                attention_mask = attention_mask.to(model.device)
                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                gen = self.tokenizer.decode(
                    out[0, input_ids.shape[1]:], skip_special_tokens=True
                ).strip()
                texts.append(gen if gen else "<EMPTY>")
        finally:
            model.config.use_cache = old_use_cache
            if was_training:
                model.train()

        print(f"[Canary epoch {state.epoch:.0f}] " + " | ".join(t[:80] for t in texts))

        if all(t == "<EMPTY>" for t in texts):
            self.stopped = True
            print("!!! Canary: ВСЕ генерации пустые — модель не обучилась.\n"
                  "    Останавливаю обучение.")
            control.should_training_stop = True


class BestEpochValMetricCallback(TrainerCallback):
    """Раз в эпоху: ЧИСТАЯ метрика (src-only генерация + матчинг)
    на N val-примерах + сохранение ЛУЧШЕГО адаптера в best_prefix.pt."""

    def __init__(self, val_ds, val_references, tokenizer, collator,
                 output_dir, task_kind="emotion_cls",
                 num_examples=200, max_new_tokens=6):
        n = min(num_examples, len(val_ds), len(val_references))
        self.loader = DataLoader(
            val_ds.select(range(n)), batch_size=8, collate_fn=collator
        )
        self.references = list(val_references[:n])
        self.tokenizer = tokenizer
        self.output_dir = output_dir
        self.task_kind = task_kind
        self.metric_name = "accuracy" if task_kind == "emotion_cls" else "rougeL"
        self.max_new_tokens = max_new_tokens
        self.rouge = evaluate.load("rouge") if task_kind != "emotion_cls" else None
        self.best = -1.0
        self.best_epoch = None
        self.history = []
        self._collapse_reported = False

    @torch.no_grad()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return

        was_training = model.training
        model.eval()
        old_use_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True

        preds = []
        try:
            for batch in self.loader:
                input_ids, attention_mask = strip_label_tokens(batch)
                input_ids = input_ids.to(model.device)
                attention_mask = attention_mask.to(model.device)
                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                for j in range(input_ids.size(0)):
                    pred = self.tokenizer.decode(
                        out[j, input_ids.shape[1]:], skip_special_tokens=True
                    ).strip()
                    preds.append(pred if pred else " ")
        finally:
            model.config.use_cache = old_use_cache
            if was_training:
                model.train()

        if self.task_kind == "emotion_cls":
            score = float(np.mean([
                match_emotion(p) == r for p, r in zip(preds, self.references)
            ]))
        else:
            score = float(self.rouge.compute(
                predictions=preds, references=self.references,
                rouge_types=["rougeL"],
            )["rougeL"])
        self.history.append({"epoch": round(state.epoch, 2), "value": score})

        if score > self.best:
            self.best = score
            self.best_epoch = round(state.epoch, 2)
            os.makedirs(self.output_dir, exist_ok=True)
            torch.save(
                {n: p.data.detach().cpu().clone()
                 for n, p in model.named_parameters() if p.requires_grad},
                os.path.join(self.output_dir, "best_prefix.pt"),
            )

        # D2: стоп коллапса генерации. Урок ARTICLE s44 (§4.16): CE-плато
        # скрывает деградацию генерации — guard по лоссу её не видит,
        # канарейка видит только ПУСТЫЕ генерации. Здесь: глубокое
        # падение val-метрики относительно лучшей = остановка.
        if (len(self.history) >= 3 and score < 0.5 * self.best
                and not self._collapse_reported):
            self._collapse_reported = True
            print(f"\n!!! Коллапс: val {self.metric_name}={score:.4f} < "
                  f"0.5 x best ({self.best:.4f}) на эпохе {state.epoch:.0f} "
                  f"— останавливаю обучение.\n")
            control.should_training_stop = True
        elif (len(self.history) >= 3 and score < 0.75 * self.best
                and not getattr(self, "_softdeg_warned", False)):
            # критика-2: мягкая деградация (0.5x < val < 0.75x best) —
            # только предупреждение: агрессивный стоп опасен (MLE s44
            # D2.1 восстановился с 0.026 на e1 до 0.176 на e2)
            self._softdeg_warned = True
            print(f"[SoftDegradation] val {self.metric_name}={score:.4f} < "
                  f"0.75 x best ({self.best:.4f}) — предупреждение, "
                  f"БЕЗ остановки")

        with open(os.path.join(self.output_dir, "val_metric_history.json"), "w") as f:
            _json.dump(
                {"metric": self.metric_name, "history": self.history,
                 "best": self.best, "best_epoch": self.best_epoch},
                f, indent=2,
            )

        print(f"[ValMetric epoch {state.epoch:.0f}] {self.metric_name}={score:.4f} | "
              f"best={self.best:.4f} @ epoch {self.best_epoch:.0f}")

    def on_train_end(self, args, state, control, **kwargs):
        print(f"[ValMetric] Итог: best {self.metric_name}={self.best:.4f} @ epoch "
              f"{self.best_epoch:.0f} -> best_prefix.pt")


def prepare_trainer(model, train_ds, val_ds, rl_ds, rl_references, tokenizer, cfg,
                    val_references=None):
    """Конфигурация тренера полностью из cfg — включая output_dir."""

    def compute_metrics(eval_pred):
        return {}

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.total_epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type=cfg.lr_scheduler,
        warmup_steps=cfg.warmup_steps,
        logging_steps=cfg.logging_steps,
        eval_strategy=cfg.eval_strategy,
        save_strategy=cfg.save_strategy,
        save_total_limit=cfg.save_total_limit,
        load_best_model_at_end=cfg.load_best_model_at_end,
        bf16=cfg.bf16,
        fp16=cfg.fp16,
        gradient_checkpointing=cfg.gradient_checkpointing,
        report_to=cfg.report_to,
        remove_unused_columns=cfg.remove_unused_columns,
        dataloader_pin_memory=cfg.dataloader_pin_memory,
        seed=cfg.seed,
        prediction_loss_only=True,
        max_grad_norm=cfg.max_grad_norm,
    )

    canary = Phase1CanaryCallback(
        canary_ds=val_ds,
        tokenizer=tokenizer,
        collator=causal_lm_collator,
        first_epoch=2,
        max_new_tokens=cfg.eval_max_new_tokens,
    )

    callbacks = [canary]

    if val_references is not None:
        callbacks.append(BestEpochValMetricCallback(
            val_ds=val_ds,
            val_references=val_references,
            tokenizer=tokenizer,
            collator=causal_lm_collator,
            output_dir=cfg.output_dir,
            task_kind=cfg.task_kind,
            num_examples=cfg.val_rouge_examples,
            max_new_tokens=cfg.eval_max_new_tokens,
        ))

    trainer = TwoPhaseTrainerGen(
        model=model,
        args=training_args,
        phase1_epochs=cfg.phase1_epochs,
        alpha=cfg.alpha,
        beta=cfg.beta,
        gamma=cfg.gamma,
        gamma_auto=cfg.gamma_auto,
        gamma_target_frac=cfg.gamma_target_frac,
        sigma=cfg.sigma,
        contrast_mode=cfg.contrast_mode,
        contrast_from_start=cfg.contrast_from_start,
        contrastive_dropout=cfg.contrastive_dropout,
        contrastive_tau=cfg.contrastive_tau,
        k_negatives=cfg.k_negatives,
        rl_interval=cfg.rl_interval,
        rl_num_batches=cfg.rl_num_batches,
        rl_subset_size=cfg.rl_subset_size,
        use_reward_baseline=cfg.use_reward_baseline,
        phase2_lr=cfg.phase2_lr,
        divergence_ce_threshold=cfg.divergence_ce_threshold,
        guard_start_step=cfg.guard_start_step,
        rl_dataset=rl_ds,
        rl_references=rl_references,
        tokenizer=tokenizer,
        task_cfg=cfg,
        contrast_neg_in_graph=getattr(cfg, "contrast_neg_in_graph", False),
        reward_baseline_beta=getattr(cfg, "reward_baseline_beta", 0.9),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=causal_lm_collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    return trainer

print("collator + strip_label_tokens + verbalizer_eval + коллбэры (чистый eval) ✓")


collator + strip_label_tokens + verbalizer_eval + коллбэры (чистый eval) ✓


In [10]:
# # ============================================================
# # CELL 8: Train!
# # ============================================================
# trainer = prepare_trainer(model, train_ds, val_ds, rl_ds, rl_references, tok, cfg)

# print(f"\n{'='*60}")
# print(f"Starting training: {cfg.total_epochs} epochs "
#       f"(Phase1: {cfg.phase1_epochs}, Phase2: {cfg.total_epochs - cfg.phase1_epochs})")
# print(f"{'='*60}\n")

# start_time = time.time()
# trainer.train()
# elapsed = time.time() - start_time

# print(f"\nTraining done in {elapsed/60:.1f} min")


In [11]:
# # ============================================================
# # CELL 10: Test evaluation with ROUGE
# # ============================================================
# from tqdm.auto import tqdm

# model.eval()

# old_use_cache = getattr(model.config, "use_cache", True)
# model.config.use_cache = True

# test_loader = DataLoader(
#     test_ds,
#     batch_size=1,
#     collate_fn=causal_lm_collator,
# )

# predictions = []
# references = []

# with torch.no_grad():
#     for i, batch in enumerate(tqdm(test_loader, total=len(test_loader))):
#         input_ids = batch["input_ids"].to(model.device)
#         attention_mask = batch["attention_mask"].to(model.device)

#         gen_ids = model.generate(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             max_new_tokens=48,
#             do_sample=False,
#             num_beams=1,
#             pad_token_id=tok.pad_token_id,
#             eos_token_id=tok.eos_token_id,
#         )

#         new_tokens = gen_ids[:, input_ids.shape[1]:]
#         pred = tok.decode(new_tokens[0], skip_special_tokens=True).strip()

#         if len(pred) == 0:
#             pred = " "

#         predictions.append(pred)
#         references.append(test_references[i])

# model.config.use_cache = old_use_cache

# rouge = evaluate.load("rouge")

# scores = rouge.compute(
#     predictions=predictions,
#     references=references,
#     rouge_types=["rouge1", "rouge2", "rougeL"],
# )

# print("=" * 50)
# print("TEST ROUGE")
# print("=" * 50)
# for k, v in scores.items():
#     print(f"{k}: {v:.4f}")

# print("\nExamples:")
# for i in range(3):
#     print("-" * 80)
#     print(f"REF: {references[i]}")
#     print(f"GEN: {predictions[i]}")

In [12]:
# ============================================================
# CELL 10b: Zero-shot референс — базовая модель без обучения,
# с тем же шаблоном, что и обучение (честное сравнение)
# ============================================================
import json
from tqdm.auto import tqdm

# Освобождаем память обученной модели (run_experiment грузит свою)
for _name in ("model", "trainer", "base"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

set_seed(cfg.seed)

zs_base = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
zs_base.eval()
zs_device = "cuda" if torch.cuda.is_available() else "cpu"
zs_base.to(zs_device)

if cfg.task_kind == "emotion_cls":
    correct = 0
    zs_examples = []
    with torch.no_grad():
        for i in tqdm(range(len(raw_test)), desc="zero-shot cls"):
            text = raw_test[i][cfg.text_column]
            label = raw_test[i][cfg.target_column]
            prompt = cfg.prompt_template.format(source=text)
            enc = tok(prompt, return_tensors="pt", truncation=True,
                      max_length=cfg.max_source_len + 16).to(zs_device)
            out = zs_base.generate(
                **enc,
                max_new_tokens=cfg.eval_max_new_tokens,
                do_sample=False,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )
            pred = tok.decode(
                out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True
            ).strip()
            pred_label = match_emotion(pred)
            correct += int(pred_label == label)
            if len(zs_examples) < 6:
                zs_examples.append((label, pred_label, pred))

    zs_scores = {"accuracy": correct / max(len(raw_test), 1)}

    print("=" * 60)
    print("ZERO-SHOT: классификация эмоции (ED, 32 класса)")
    print("=" * 60)
    print(f"accuracy: {zs_scores['accuracy']:.4f}")
    print(f"random guess: 1/32 = {1/32:.4f}")
    print("\nРеференсы из статьи (их Table 1, EmpatheticDialogues):")
    print("  HandCraft:  RoBERTa 0.272 / Falcon 0.203")
    print("  Prompt v2:  RoBERTa 0.454 / Falcon 0.391")
    print("  CRL-Prompt: RoBERTa 0.474 / Falcon 0.402")
    print("=" * 60)
    print("\nПримеры (ref -> pred | сырая генерация):")
    for label, pl, p in zs_examples:
        print(f"  {label:<14} -> {str(pl):<14} | {p[:30]!r}")
else:
    zs_rouge = evaluate.load("rouge")
    zs_preds, zs_refs = [], []
    with torch.no_grad():
        for i in tqdm(range(len(test_ds)), desc="zero-shot"):
            doc = raw_test[i][cfg.text_column]
            prompt = cfg.prompt_template.format(source=doc)
            enc = tok(prompt, return_tensors="pt", truncation=True,
                      max_length=cfg.max_source_len + 16).to(zs_device)
            out = zs_base.generate(
                **enc,
                max_new_tokens=cfg.eval_max_new_tokens,
                do_sample=False,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )
            pred = tok.decode(
                out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True
            ).strip()
            zs_preds.append(pred if pred else " ")
            zs_refs.append(test_references[i])

    zs_scores = zs_rouge.compute(
        predictions=zs_preds,
        references=zs_refs,
        rouge_types=["rouge1", "rouge2", "rougeL"],
    )
    print("=" * 50)
    print("ZERO-SHOT REFERENCE (без префикса, с инструкцией)")
    print("=" * 50)
    for k, v in zs_scores.items():
        print(f"{k}: {v:.4f}")
    print("=" * 50)
    print("Обученный префикс должен быть ЗНАЧИМО выше этой планки.\n")
    print("Examples:")
    for i in range(3):
        print("-" * 80)
        print(f"REF: {zs_refs[i]}")
        print(f"GEN: {zs_preds[i]}")

# Сохраняем рядом с остальными результатами
os.makedirs(cfg.output_dir, exist_ok=True)
with open(f"{cfg.output_dir}/zeroshot_results.json", "w") as f:
    json.dump({"experiment": "zero-shot", "task_kind": cfg.task_kind,
               "scores": zs_scores}, f, indent=2)

# Освобождаем память
del zs_base
gc.collect()
torch.cuda.empty_cache()


zero-shot cls: 100%|██████████| 500/500 [01:08<00:00,  7.34it/s]


ZERO-SHOT: классификация эмоции (ED, 32 класса)
accuracy: 0.2580
random guess: 1/32 = 0.0312

Референсы из статьи (их Table 1, EmpatheticDialogues):
  HandCraft:  RoBERTa 0.272 / Falcon 0.203
  Prompt v2:  RoBERTa 0.454 / Falcon 0.391
  CRL-Prompt: RoBERTa 0.474 / Falcon 0.402

Примеры (ref -> pred | сырая генерация):
  guilty         -> guilty         | 'I felt guilty when I was'
  caring         -> sad            | 'I felt sad and touched by'
  lonely         -> None           | 'I feel so empty.\nSit'
  excited        -> excited        | 'I am excited.\nQuestion:'
  sad            -> sad            | 'sadness\nQuestion: How would'
  caring         -> None           | 'I feel bad for him'


# Baseline

In [11]:
# ============================================================
# КОНФИГИ: D1.6 — осмысленные vs вырожденные пары на чистом протоколе
# Все наследуют CFG (contrast_from_start=True, одна фаза, batch 8)
# ============================================================
class CFG_MLE_Only(CFG):
    output_dir = "./gen_crl_mle_baseline"

    beta = 0.0
    gamma = 0.0
    gamma_auto = False


class CFG_ContrastMask(CFG):
    """Контраст с негативами-масками промпта (рецепт статьи),
    с 1-й эпохи, смягчённые значения (β=0.1, τ=0.5)"""
    output_dir = "./gen_dialogue_ed_contrast_mask"

    beta = 0.1
    contrast_mode = "mask"
    gamma = 0.0
    gamma_auto = False


class CFG_ContrastWrongLabel(CFG):
    """Ядро темы: жёсткие негативы (ситуация, путаемая эмоция)"""
    output_dir = "./gen_dialogue_ed_contrast_wl"

    beta = 0.1
    contrast_mode = "wrong_label"
    gamma = 0.0
    gamma_auto = False


class CFG_ContrastMaskArticle(CFG):
    """Критика: тестируем и ЗНАЧЕНИЯ СТАТЬИ (β=0.3, τ=0.1), а не только
    ослабленные. На классификации они не проверялись (в иссл.-1
    проверялись на генерации XSum, где разрушали её)."""
    output_dir = "./gen_dialogue_ed_contrast_mask_article"

    beta = 0.3
    contrast_mode = "mask"
    contrastive_tau = 0.1
    gamma = 0.0
    gamma_auto = False


class CFG_RL_Only(CFG):
    """Только RL (REINFORCE), без контраста. β=0, γ>0.
    ОДНОФАЗНО (унаследовано 4=4): RL-события каждые rl_interval шагов
    С САМОГО НАЧАЛА, phase2_lr НЕ применяется (RL-кики на полном lr
    до 5e-4). Это осознанный выбор по уроку §4.6: эффект «только в
    фазе 2» не доживает до best-epoch отбора (1/24). Отличие от
    RL-прогонов D1.2/D1.3 (там RL лишь в 5-й эпохе на lr 1e-4)."""
    output_dir = "./gen_dialogue_ed_rl_only"

    beta = 0.0
    contrast_mode = "mask"          # не используется при β=0
    gamma = 2e-5                    # стартовое; auto-калибровка перепишет
    gamma_auto = True               # калибровка по первому RL-событию
    # остальные параметры (sigma, rl_interval, ...) наследуются из CFG


class CFG_Full_CRL(CFG):
    """Full CRL = Contrast(mask) + RL. β=0.1, τ=0.5, γ>0.
    Значения контраста == CFG_ContrastMask -> парная разность
    Full-CRL − contrast-mask изолирует вклад RL. Однофазно: контраст
    (contrast_from_start) и RL активны с 1-й эпохи."""
    output_dir = "./gen_dialogue_ed_full_crl"

    beta = 0.1
    contrast_mode = "mask"
    contrastive_tau = 0.5
    gamma = 2e-5
    gamma_auto = True

cfg_baseline = CFG_MLE_Only()
cfg_mask = CFG_ContrastMask()
cfg_wl = CFG_ContrastWrongLabel()
cfg_mask_article = CFG_ContrastMaskArticle()
cfg_rl = CFG_RL_Only()
cfg_full = CFG_Full_CRL()


print("=" * 80)
print("EXPERIMENT CONFIGS (D1.6, Qwen2.5-3B, чистый eval)")
print("=" * 80)
for c, name in [(cfg_baseline, "MLE-only"),
                (cfg_mask, "contrast-mask (β=0.1,τ=0.5)"),
                (cfg_wl, "contrast-wronglabel (hard)"),
                (cfg_mask_article, "contrast-mask-ARTICLE (β=0.3,τ=0.1)"),
                (cfg_rl, "RL-only (γ=2e-5)"),            
                (cfg_full, "Full CRL (mask+RL)")]:
    print(f"{name:<34} mode={c.contrast_mode:<12} beta={c.beta:<5} "
          f"tau={c.contrastive_tau:<5} -> {c.output_dir}")
print("=" * 80)


EXPERIMENT CONFIGS (D1.6, Qwen2.5-3B, чистый eval)
MLE-only                           mode=mask         beta=0.0   tau=0.5   -> ./gen_crl_mle_baseline
contrast-mask (β=0.1,τ=0.5)        mode=mask         beta=0.1   tau=0.5   -> ./gen_dialogue_ed_contrast_mask
contrast-wronglabel (hard)         mode=wrong_label  beta=0.1   tau=0.5   -> ./gen_dialogue_ed_contrast_wl
contrast-mask-ARTICLE (β=0.3,τ=0.1) mode=mask         beta=0.3   tau=0.1   -> ./gen_dialogue_ed_contrast_mask_article
RL-only (γ=2e-5)                   mode=mask         beta=0.0   tau=0.5   -> ./gen_dialogue_ed_rl_only
Full CRL (mask+RL)                 mode=mask         beta=0.1   tau=0.5   -> ./gen_dialogue_ed_full_crl


In [13]:
# ============================================================
# COMPARATIVE EXPERIMENT — исследование-2, итерация D1.6
# Чистый протокол + 3B + парный бутстреп по примерам.
# RESUME-SAFE: читает summary_d1_6.json, дозапускает неполные сиды.
# ============================================================
import json
import copy
from collections import defaultdict


def select_best_prefix_init(model, train_ds, val_ds, tok, cfg,
                            n_inits=4, probe_steps=200, n_val=200):
    """Probe-отбор = ГЕНЕРАТОР общего старта. Модель на GPU ДО вызова."""
    prompt_params = [(n, p) for n, p in model.named_parameters()
                     if p.requires_grad and "prompt_encoder" in n]
    if not prompt_params:
        print("[InitSel] prompt_encoder не найден — пропуск отбора")
        return

    device = model.device
    accum = cfg.gradient_accumulation_steps
    probe_lr = cfg.learning_rate
    autocast_ctx = lambda: torch.autocast("cuda", dtype=torch.bfloat16,
                                          enabled=cfg.bf16 and device.type == "cuda")

    val_loader = DataLoader(
        val_ds.select(range(min(n_val, len(val_ds)))),
        batch_size=cfg.batch_size, collate_fn=causal_lm_collator,
    )

    @torch.no_grad()
    def val_ce():
        model.eval()
        total, ntok = 0.0, 0
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()
                     if k in ("input_ids", "attention_mask", "labels")}
            with autocast_ctx():
                out = model(input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"],
                            labels=batch["labels"])
            n_t = int((batch["labels"] != -100).sum())
            total += out.loss.item() * n_t
            ntok += n_t
        model.train()
        return total / max(ntok, 1)

    def reinit_prompt():
        enc = getattr(model, "prompt_encoder", None)
        assert enc is not None, "model.prompt_encoder не найден"
        for m in enc.modules():
            if hasattr(m, "reset_parameters"):
                m.reset_parameters()
        for _, p in prompt_params:
            p.data = p.data.float()

    print(f"\n[InitSel] best-of-{n_inits} | probe {probe_steps} опт. шагов | "
          f"val CE на {min(n_val, len(val_ds))} примерах | {device}")

    candidates = []
    for i in range(n_inits):
        candidate_seed = cfg.seed * 1000 + i
        set_seed(candidate_seed)
        reinit_prompt()

        opt = torch.optim.AdamW([p for _, p in prompt_params], lr=probe_lr)
        opt.zero_grad(set_to_none=True)

        probe_loader = DataLoader(
            train_ds, batch_size=cfg.batch_size, shuffle=True,
            collate_fn=causal_lm_collator,
            generator=torch.Generator().manual_seed(candidate_seed),
        )
        probe_steps_done = 0
        for batch in probe_loader:
            if probe_steps_done >= probe_steps:
                break
            batch = {k: v.to(device) for k, v in batch.items()
                     if k in ("input_ids", "attention_mask", "labels")}
            with autocast_ctx():
                out = model(input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"],
                            labels=batch["labels"])
            (out.loss / accum).backward()
            probe_steps_done += 1
            if probe_steps_done % accum == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for _, p in prompt_params], cfg.max_grad_norm)
                opt.step()
                opt.zero_grad(set_to_none=True)

        ce = val_ce()
        snap = {n: p.data.detach().cpu().clone() for n, p in prompt_params}
        candidates.append((ce, i, snap))
        print(f"[InitSel] кандидат {i}: val CE = {ce:.4f}")

    candidates.sort(key=lambda r: r[0])
    best_ce, best_i, best_snap = candidates[0]
    with torch.no_grad():
        for n, p in prompt_params:
            p.data.copy_(best_snap[n].to(p.device))

    ranking = " | ".join(f"#{i}: {ce:.3f}" for ce, i, _ in candidates)
    print(f"[InitSel] выбран кандидат {best_i} (val CE {best_ce:.4f}); "
          f"ранжирование: {ranking}\n")


def run_experiment(cfg, experiment_name, init_state=None, return_init_state=False):
    """init_state — общий старт пары; return_init_state=True возвращает
    (scores, snapshot). Сохраняет per-example корректности (для
    парного бутстрепа) и вербализатор-скоринг."""

    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT: {experiment_name}")
    print(f"{'='*70}\n")

    gc.collect()
    torch.cuda.empty_cache()
    set_seed(cfg.seed)

    dtype = torch.bfloat16 if cfg.bf16 else torch.float32
    base = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        dtype=dtype,
        trust_remote_code=cfg.trust_remote_code,
    )

    if cfg.gradient_checkpointing:
        base.gradient_checkpointing_enable()
    base.config.use_cache = cfg.use_cache

    peft_cfg = PrefixTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=cfg.num_virtual_tokens,
        prefix_projection=cfg.prefix_projection,
        inference_mode=False,
    )

    model = get_peft_model(base, peft_cfg)
    del base

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    model.enable_input_require_grads()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    prompt_param_list = [(n, p) for n, p in model.named_parameters()
                         if p.requires_grad and "prompt_encoder" in n]

    start_snapshot = None
    if init_state is not None:
        with torch.no_grad():
            for n, p in prompt_param_list:
                if n in init_state:
                    p.data.copy_(init_state[n].to(p.device))
        print(f"[PairedStart] загружен ОБЩИЙ старт ({len(init_state)} тензоров "
              f"префикса) — probe пропущен\n")
    elif getattr(cfg, "init_selection_inits", 0) > 0:
        select_best_prefix_init(
            model, train_ds, val_ds, tok, cfg,
            n_inits=cfg.init_selection_inits,
            probe_steps=cfg.init_selection_probe_steps,
        )
        if return_init_state:
            start_snapshot = {n: p.data.detach().cpu().clone()
                              for n, p in prompt_param_list}

    print("Trainable parameters:")
    model.print_trainable_parameters()

    trainer = prepare_trainer(
        model, train_ds, val_ds, rl_ds, rl_references, tok, cfg,
        val_references=val_references,
    )

    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time
    rl_hist = list(getattr(trainer, "rl_reward_history", []) or [])

    print(f"\n{experiment_name} completed in {elapsed/60:.1f} min")

    # Подгружаем ЛУЧШИЙ по val-метрике адаптер
    val_metric_info = {}
    best_path = os.path.join(cfg.output_dir, "best_prefix.pt")
    hist_path = os.path.join(cfg.output_dir, "val_metric_history.json")
    if os.path.exists(best_path):
        best_state = torch.load(best_path, map_location="cpu")
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best_state:
                    p.data.copy_(best_state[n].to(p.device))
        if os.path.exists(hist_path):
            with open(hist_path) as f:
                val_metric_info = json.load(f)
            print(f"Loaded best adapter: val {val_metric_info['metric']}"
                  f"={val_metric_info['best']:.4f} "
                  f"@ epoch {val_metric_info['best_epoch']:.0f} "
                  f"(история: {[round(h['value'], 4) for h in val_metric_info['history']]})")
        else:
            print("Loaded best adapter (best_prefix.pt)")
    else:
        print("best_prefix.pt не найден — оцениваю последнюю эпоху")

    # ---------- ЧИСТЫЙ тест (src-only generate, без золотой метки) ----------
    print(f"\nEvaluating {experiment_name} on test set (clean protocol)...")
    model.eval()

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        collate_fn=causal_lm_collator,
    )

    predictions = []
    references = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask = strip_label_tokens(batch)
            input_ids = input_ids.to(model.device)
            attention_mask = attention_mask.to(model.device)

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=cfg.eval_max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )

            for j in range(input_ids.size(0)):
                pred = tok.decode(
                    gen_ids[j, input_ids.shape[1]:], skip_special_tokens=True
                ).strip()
                predictions.append(pred if pred else " ")

    references = list(test_references)

    per_ex = None
    if cfg.task_kind == "emotion_cls":
        pred_labels = [match_emotion(p) for p in predictions]
        correct_vec = [int(pl == r) for pl, r in zip(pred_labels, references)]
        scores = {"accuracy": float(np.mean(correct_vec))}

        # Вербализатор-скоринг (протокол статьи, детерминированный)
        verb_acc, verb_correct = verbalizer_eval(
            model, test_ds, references, tok)
        scores["verb_accuracy"] = verb_acc
    else:
        rouge = evaluate.load("rouge")
        scores = rouge.compute(
            predictions=predictions,
            references=references,
            rouge_types=["rouge1", "rouge2", "rougeL"],
        )
        # D2: per-example ROUGE-L (для парного бутстрепа) + Distinct-1/2
        per_ex = [float(x) for x in rouge.compute(
            predictions=predictions, references=references,
            rouge_types=["rougeL"], use_aggregator=False)["rougeL"]]
        scores["distinct1"], scores["distinct2"] = distinct_n(predictions)
        correct_vec = None
        verb_correct = None
        pred_labels = None

    print(f"\n{'='*70}")
    print(f"RESULTS: {experiment_name}")
    print(f"{'='*70}")
    for k, v in scores.items():
        print(f"{k}: {v:.4f}")
    print(f"{'='*70}\n")

    if pred_labels is not None:
        miss = [(r, pl, p) for r, pl, p in zip(references, pred_labels, predictions)
                if pl != r][:6]
        print("Примеры ошибок (ref -> pred | генерация):")
        for r, pl, p in miss:
            print(f"  {r:<14} -> {str(pl):<14} | {p[:30]!r}")
    else:
        print("Примеры генераций (test, чистый вход):")
        for r, p in list(zip(references, predictions))[:4]:
            print(f"  REF: {r[:90]!r}")
            print(f"  GEN: {p[:90]!r}")

    results = {
        "experiment": experiment_name,
        "config": {
            "task_kind": cfg.task_kind,
            "model": cfg.model_name,
            "beta": cfg.beta,
            "contrast_mode": getattr(cfg, "contrast_mode", None),
            "contrastive_tau": cfg.contrastive_tau,
            "contrast_from_start": getattr(cfg, "contrast_from_start", False),
            "epochs": cfg.total_epochs,
            "lr": cfg.learning_rate,
            "batch": cfg.batch_size,
            "seed": cfg.seed,
            "paired_common_start": init_state is not None,
            "clean_eval": True,
        },
        "test_scores": scores,
        "correct_vec": correct_vec,
        "verb_correct_vec": verb_correct,
        "per_example_rougeL": per_ex,
        "predictions": predictions,
        "rl_reward_history": rl_hist,
        "val_metric": val_metric_info,
        "training_time_min": elapsed / 60,
    }

    output_file = f"{cfg.output_dir}/results.json"
    os.makedirs(cfg.output_dir, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {output_file}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    if return_init_state:
        return scores, start_snapshot
    return scores

# ============================================================
# ЗАПУСК D1.6 (RESUME-SAFE)
# 7 сидов × {MLE, mask, wronglabel, RL-only, Full-CRL} + 3 сида
# mask-ARTICLE (значения статьи; расходящиеся умирают за ~5 мин).
# Время (замер): MLE 9.7 + probe ~3.5, контраст 13.9, RL ~18,
# Full ~22 мин; полные сиды ~75 мин, 45/46 ~50 мин -> ~3.5 ч.
# ВАЖНО: старые общие старты (MLE/mask/wl) НЕ сохранялись на диск,
# а парность требует общего старта -> ВСЕ 5 конфигов перезапускаются
# с НОВЫМ стартом. С этого прогона shared_init.pt пишется на диск:
# будущие добавления конфигов не потребуют перезапуска MLE.
# ============================================================
# 7 сидов (расширение с 3 до 7 для мощности; решение зафиксировано
# после гипотез, все новые сиды включаются без отбора). 45/46:
# MLE/RL/старт уже есть с D1.6b -> добираются только mask/wl/Full.
SEEDS = [42, 43, 44, 45, 46, 47, 48]
D16_CONFIGS = [
    (cfg_mask, "contrast-mask"),
    (cfg_wl, "contrast-wronglabel"),
    (cfg_rl,   "RL-only"),           
    (cfg_full, "Full-CRL"),   
]
NEEDED_KEYS = {"seed", "MLE-only", "contrast-mask", "contrast-wronglabel",
               "RL-only", "Full-CRL"}

_summary_path = f"{cfg.output_dir}/summary_d1_6.json"
paired_d16 = []
if os.path.exists(_summary_path):
    with open(_summary_path) as f:
        _saved = json.load(f)
    paired_d16 = _saved.get("rows", _saved.get("per_seed", []))

done_seeds = {r["seed"] for r in paired_d16 if NEEDED_KEYS.issubset(r)}
todo_seeds = [s for s in SEEDS if s not in done_seeds]
print(f"D1.6 resume: готово {sorted(done_seeds)}, к запуску {todo_seeds}")
if not todo_seeds:
    print("Все сиды завершены — сразу агрегация.\n")

for seed in todo_seeds:
    c_mle = copy.copy(cfg_baseline)
    c_mle.seed = seed
    c_mle.output_dir = f"{cfg_baseline.output_dir}_d16_seed{seed}"
    shared_path = os.path.join(c_mle.output_dir, "shared_init.pt")
    _mle_json = os.path.join(c_mle.output_dir, "results.json")
    if os.path.exists(shared_path) and os.path.exists(_mle_json):
        # общий старт и MLE завершены в прошлом прогоне сида —
        # добираем только недостающие конфиги (парность сохранена)
        shared = torch.load(shared_path, map_location="cpu")
        with open(_mle_json) as f:
            scores_mle = json.load(f)["test_scores"]
        print(f"[Resume] сид {seed}: общий старт и MLE взяты с диска")
    else:
        scores_mle, shared = run_experiment(
            c_mle, f"MLE-only D1.6 (seed {seed})", return_init_state=True)
        os.makedirs(c_mle.output_dir, exist_ok=True)
        torch.save(shared, shared_path)
    assert shared is not None, "общий старт не создан (probe не работал?)"

    row = {"seed": seed, "MLE-only": scores_mle["accuracy"],
           "MLE-only-verb": scores_mle["verb_accuracy"]}

    for base_cfg, name in D16_CONFIGS:
        c = copy.copy(base_cfg)
        c.seed = seed
        c.output_dir = f"{base_cfg.output_dir}_d16_seed{seed}"
        s = run_experiment(c, f"{name} D1.6 (seed {seed}, общий старт)",
                           init_state=shared)
        row[name] = s["accuracy"]
        row[f"{name}-verb"] = s["verb_accuracy"]

    paired_d16 = [r for r in paired_d16 if r["seed"] != seed] + [row]
    del shared
    gc.collect()
    torch.cuda.empty_cache()

    with open(_summary_path, "w") as f:
        json.dump({"rows": paired_d16}, f, indent=2)
    print(f"\n>>> Сид {seed} завершён: {row}\n")

# Отдельные прогоны: значения статьи (β=0.3, τ=0.1) на 3 сидах.
# Сид 42 остановлен guard'ом (расходимость на шаге 1027) — сиды 43/44
# добавлены, чтобы показать систематичность (дёшево: расходящиеся
# прогоны живут ~5 мин). Сид 42 уже записан в легаси-путь без
# суффикса — переиспользуем, не перезапускаем.
art_scores_all = {}
for _s in (42, 43, 44):
    if _s == 42 and os.path.exists("./gen_dialogue_ed_contrast_mask_article/results_d16.json"):
        _dir = "./gen_dialogue_ed_contrast_mask_article"
        _path = f"{_dir}/results_d16.json"
    else:
        _dir = f"./gen_dialogue_ed_contrast_mask_article_seed{_s}"
        _path = f"{_dir}/results_d16.json"
    if not os.path.exists(_path):
        c_art = copy.copy(cfg_mask_article)
        c_art.seed = _s
        c_art.output_dir = _dir
        _sc = run_experiment(c_art, f"contrast-mask-ARTICLE D1.6 (seed {_s})")
        with open(_path, "w") as f:
            json.dump(_sc, f, indent=2)
    else:
        with open(_path) as f:
            _sc = json.load(f)
    art_scores_all[_s] = _sc

# ============================================================
# АГРЕГАЦИЯ D1.6: mean±std + парный t + ПАРНЫЙ БУТСТРЕП по примерам
# ============================================================
from math import sqrt

names = ["MLE-only"] + [n for _, n in D16_CONFIGS]

print("\n" + "=" * 80)
print(f"D1.6: чистый протокол, Qwen2.5-3B, {len(paired_d16)} сидов, общий старт (accuracy)")
print("=" * 80)
print(f"{'Config':<22}{'gen mean':<10}{'gen std':<10}{'verb mean':<11}{'per-seed (gen)'}")
print("-" * 80)
agg = {}
for n in names:
    vals = [r[n] for r in paired_d16]
    vvals = [r[f"{n}-verb"] for r in paired_d16]
    agg[n] = vals
    print(f"{n:<22}{np.mean(vals):<10.4f}{np.std(vals):<10.4f}"
          f"{np.mean(vvals):<11.4f}{[round(v, 4) for v in vals]}")
print("-" * 80)
for _s, _sc in art_scores_all.items():
    # history берём из val_metric_history.json: results_d16.json хранит
    # только {accuracy, verb_accuracy}, без val_metric
    _dir_a = ("./gen_dialogue_ed_contrast_mask_article" if _s == 42
              else f"./gen_dialogue_ed_contrast_mask_article_seed{_s}")
    try:
        with open(os.path.join(_dir_a, "val_metric_history.json")) as f:
            _hist_a = json.load(f)["history"]
    except FileNotFoundError:
        _hist_a = []
    _note = (f"остановлен на e{len(_hist_a)}/{cfg.total_epochs} (guard)"
             if _hist_a and len(_hist_a) < cfg.total_epochs else "все эпохи")
    print(f"mask-ARTICLE (β=0.3,τ=0.1, seed {_s}): gen={_sc['accuracy']:.4f} "
          f"verb={_sc['verb_accuracy']:.4f} [{_note}]")
if "zs_scores" in globals():
    print(f"zero-shot: {zs_scores['accuracy']:.4f} | random: {1/32:.4f}")
print("ВАЖНО: числа НЕ сопоставимы с D1.1b-D1.5 (там был баг эха — вход")
print("содержал золотую метку). Прошлые 'accuracy' были эхо-рейтом.")
print("=" * 80)

# t-критерий: df = n_сидов − 1. Прежний словарь был ошибочен:
# ключ трактовался как df при обращении по числу сидов (для
# n=3 брали 3.182 вместо 4.303). Все выводы «н.з.» в D1.x
# сохраняются — правильный порог только строже.
t_crit = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365}  # n_сидов -> t(0.05, df=n-1)

# Парный бутстреп по примерам: читаем correct_vec из results.json
def load_vec(config_name, seed):
    base_dirs = {
        "MLE-only": cfg_baseline.output_dir,
        "contrast-mask": cfg_mask.output_dir,
        "contrast-wronglabel": cfg_wl.output_dir,    
        "RL-only":               cfg_rl.output_dir,      
        "Full-CRL":              cfg_full.output_dir,  
    }
    path = f"{base_dirs[config_name]}_d16_seed{seed}/results.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f).get("correct_vec")
    return None

def paired_bootstrap(a, b, label_a, label_b, n_boot=10000):
    """Парный бутстреп по (seed, пример): бесплатная стат. мощность
    против t-критерия на n=3 (критика). Возвращает CI и p."""
    diffs = []
    for r in paired_d16:
        va, vb = load_vec(a, r["seed"]), load_vec(b, r["seed"])
        if va is None or vb is None:
            continue
        diffs.extend([x - y for x, y in zip(va, vb)])
    if not diffs:
        print(f"  {label_a} − {label_b}: correct_vec не найден")
        return
    d = np.array(diffs, dtype=float)
    rng = np.random.default_rng(42)
    boots = np.array([d[rng.integers(0, len(d), len(d))].mean()
                      for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    print(f"  {label_a} − {label_b}: {d.mean():+.4f} "
          f"[бутстреп 95% CI: {lo:+.4f}, {hi:+.4f}], p={p:.4f} "
          f"({'ЗНАЧИМО' if p < 0.05 else 'не значимо'}), n={len(d)} пар")

print("\nПарные разности:")
print("а) t-критерий по сидам (n=3):")
for a, b in [("contrast-mask", "MLE-only"),
             ("contrast-wronglabel", "MLE-only"),
             ("contrast-wronglabel", "contrast-mask"),
             ("RL-only", "MLE-only"),
             ("Full-CRL", "MLE-only"),
             ("Full-CRL", "contrast-mask")]:
    d = [r[a] - r[b] for r in paired_d16]
    se = np.std(d, ddof=1) / sqrt(len(d)) if len(d) > 1 else float("nan")
    t = np.mean(d) / se if se and se > 0 else float("nan")
    print(f"  {a} − {b}: {np.mean(d):+.4f} ± {np.std(d):.4f} | t={t:+.2f} | "
          f"per-seed {[f'{x:+.3f}' for x in d]}")

print("\nб) парный бутстреп по примерам (все сиды):")
paired_bootstrap("contrast-mask", "MLE-only", "mask", "MLE")
paired_bootstrap("contrast-wronglabel", "MLE-only", "wronglabel", "MLE")
paired_bootstrap("contrast-wronglabel", "contrast-mask", "wronglabel", "mask")
paired_bootstrap("RL-only", "MLE-only", "RL", "MLE")
paired_bootstrap("Full-CRL", "MLE-only", "full", "MLE")
paired_bootstrap("Full-CRL", "contrast-mask", "full", "mask")

# в) ТРАЕКТОРИИ (чистый val по эпохам) + фиксированная последняя эпоха
# (урок от критика по D1.5: mask менял ТРАЕКТОРИЮ, а не пик; best-epoch
# протокол такое маскирует. Здесь метрика ЧИСТАЯ — если паттерн
# "MLE падает / mask растёт" воспроизведётся, это уже не артефакт эха)
def load_hist(config_name, seed):
    base_dirs = {
        "MLE-only": cfg_baseline.output_dir,
        "contrast-mask": cfg_mask.output_dir,
        "contrast-wronglabel": cfg_wl.output_dir,
        "RL-only":               cfg_rl.output_dir,     
        "Full-CRL":              cfg_full.output_dir,   
    }
    path = f"{base_dirs[config_name]}_d16_seed{seed}/val_metric_history.json"
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)
        return [x["value"] for x in h["history"]]
    return None

print("\nв) Траектории чистого val по эпохам:")
for n in names:
    for r in paired_d16:
        h = load_hist(n, r["seed"])
        if h:
            arrow = " -> ".join(f"{v:.3f}" for v in h)
            print(f"  {n:<22}s{r['seed']}: {arrow}")

print("\nг) Парные разности на ФИКСИРОВАННОЙ последней эпохе (бюджетный")
print("   протокол, без раннего стопа) против best-epoch (пиковый):")
for a, b in [("contrast-mask", "MLE-only"),
             ("contrast-wronglabel", "MLE-only"),
             ("contrast-wronglabel", "contrast-mask"),
             ("RL-only", "MLE-only"),
             ("Full-CRL", "MLE-only"),
             ("Full-CRL", "contrast-mask")]:
    d_last, d_best = [], []
    for r in paired_d16:
        ha, hb = load_hist(a, r["seed"]), load_hist(b, r["seed"])
        if ha and hb:
            d_last.append(ha[-1] - hb[-1])
    if d_last:
        verdict = "3/3 в плюс" if all(x > 0 for x in d_last) else "смешанно"
        print(f"  {a} − {b} @last: {np.mean(d_last):+.4f} "
              f"| per-seed {[f'{x:+.3f}' for x in d_last]} | {verdict}")

final_d16 = {
    "per_seed": paired_d16,
    "mean": {n: float(np.mean(agg[n])) for n in names},
    "std": {n: float(np.std(agg[n])) for n in names},
    "article": {str(s): {k: float(v) for k, v in sc.items()
                         if isinstance(v, float)}
                for s, sc in art_scores_all.items()},
}
with open(_summary_path, "w") as f:
    json.dump(final_d16, f, indent=2)
print(f"\nSummary saved to {_summary_path}")


D1.6 resume: готово [42, 43, 44, 45, 46, 47, 48], к запуску []
Все сиды завершены — сразу агрегация.


D1.6: чистый протокол, Qwen2.5-3B, 7 сидов, общий старт (accuracy)
Config                gen mean  gen std   verb mean  per-seed (gen)
--------------------------------------------------------------------------------
MLE-only              0.6183    0.0103    0.6174     [0.63, 0.622, 0.6, 0.628, 0.62, 0.606, 0.622]
contrast-mask         0.6277    0.0077    0.6266     [0.616, 0.622, 0.638, 0.63, 0.628, 0.622, 0.638]
contrast-wronglabel   0.5689    0.1112    0.5643     [0.622, 0.624, 0.622, 0.634, 0.604, 0.576, 0.3]
RL-only               0.6257    0.0060    0.6266     [0.632, 0.63, 0.622, 0.632, 0.624, 0.626, 0.614]
Full-CRL              0.6191    0.0118    0.6157     [0.61, 0.626, 0.608, 0.6, 0.63, 0.63, 0.63]
--------------------------------------------------------------------------------
mask-ARTICLE (β=0.3,τ=0.1, seed 42): gen=0.5960 verb=0.5960 [остановлен на e2/4 (guard)]
mask-ARTIC

In [15]:
# ============================================================
# D1.6b (подтверждение RL): пара {MLE-only, RL-only} на добавочных
# сидах 45-46 -> n=5 для разности RL − MLE.
# Мотивация (§4.16): RL − MLE = +0.0107 на 3/3 сидов (t=+1.80,
# p_boot=0.179) — единственный последовательно-положительный эффект
# за оба исследования. При сохранении величины/разброса n=5 даёт
# t≈2.3-3.0 против t_crit=2.776 (df=4). Продлеваем ТОЛЬКО пару:
# вопрос — вклад RL, mask/wl/Full к нему не относятся.
# Парность та же: MLE делает probe -> ОБЩИЙ старт (shared_init.pt),
# RL стартует с него. Папки — в соглашении _d16_seed{N} (если матрицу
# когда-нибудь расширят на 45/46, прогоны переиспользуются).
# Resume-safe; итог в ОТДЕЛЬНОМ summary_d1_6b.json (главный summary
# не трогаем — его агрегация ждёт полные строки 5 конфигов).
# Время: ~27 мин/сид (probe ~3.5 + MLE 9.7 + RL 13.4) -> ~55 мин.
# ============================================================
import copy
from math import sqrt

EXTRA_SEEDS = [45, 46]

if "paired_d16" not in globals():
    with open(f"{cfg.output_dir}/summary_d1_6.json") as f:
        paired_d16 = json.load(f)["per_seed"]

_summary_b_path = f"{cfg.output_dir}/summary_d1_6b.json"
paired_d16b = []
if os.path.exists(_summary_b_path):
    with open(_summary_b_path) as f:
        paired_d16b = json.load(f).get("rows", [])

for seed in EXTRA_SEEDS:
    if any(r["seed"] == seed for r in paired_d16b):
        print(f"[Resume] сид {seed} уже завершён")
        continue

    c_mle = copy.copy(cfg_baseline)
    c_mle.seed = seed
    c_mle.output_dir = f"{cfg_baseline.output_dir}_d16_seed{seed}"
    shared_path = os.path.join(c_mle.output_dir, "shared_init.pt")
    _mle_json = os.path.join(c_mle.output_dir, "results.json")

    if os.path.exists(shared_path) and os.path.exists(_mle_json):
        shared = torch.load(shared_path, map_location="cpu")
        with open(_mle_json) as f:
            scores_mle = json.load(f)["test_scores"]
        print(f"[Resume] сид {seed}: общий старт и MLE с диска")
    else:
        scores_mle, shared = run_experiment(
            c_mle, f"MLE-only D1.6b (seed {seed})", return_init_state=True)
        os.makedirs(c_mle.output_dir, exist_ok=True)
        torch.save(shared, shared_path)
    assert shared is not None, "общий старт не создан"

    c_rl = copy.copy(cfg_rl)
    c_rl.seed = seed
    c_rl.output_dir = f"{cfg_rl.output_dir}_d16_seed{seed}"
    _rl_json = os.path.join(c_rl.output_dir, "results.json")
    if os.path.exists(_rl_json):
        with open(_rl_json) as f:
            scores_rl = json.load(f)["test_scores"]
        print(f"[Resume] сид {seed}: RL-only с диска")
    else:
        scores_rl = run_experiment(
            c_rl, f"RL-only D1.6b (seed {seed}, общий старт)",
            init_state=shared)

    row = {"seed": seed,
           "MLE-only": scores_mle["accuracy"],
           "MLE-only-verb": scores_mle["verb_accuracy"],
           "RL-only": scores_rl["accuracy"],
           "RL-only-verb": scores_rl["verb_accuracy"]}
    paired_d16b = [r for r in paired_d16b if r["seed"] != seed] + [row]

    del shared
    gc.collect()
    torch.cuda.empty_cache()
    with open(_summary_b_path, "w") as f:
        json.dump({"rows": paired_d16b}, f, indent=2)
    print(f">>> Сид {seed} завершён: {row}\n")

# ---------- Агрегация пары: n = 3 (D1.6) + 2 (D1.6b) ----------
_have = {r["seed"] for r in paired_d16}
pair_rows = paired_d16 + [r for r in paired_d16b if r["seed"] not in _have]

def _correct_vec(base_dir, seed):
    p = f"{base_dir}_d16_seed{seed}/results.json"
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f).get("correct_vec")
    return None

print("\n" + "=" * 76)
print(f"D1.6+D1.6b: RL-only − MLE-only, n={len(pair_rows)} сидов (парный, общий старт)")
print("=" * 76)
d = [r["RL-only"] - r["MLE-only"] for r in pair_rows]
n = len(d)
se = np.std(d, ddof=1) / sqrt(n)
t = np.mean(d) / se
t_crit = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365}[n]  # df = n-1
per_seed = [f"s{r['seed']}: {r['RL-only'] - r['MLE-only']:+.3f}" for r in pair_rows]
print(f"per-seed: {per_seed}")
print(f"разность: {np.mean(d):+.4f} ± {np.std(d, ddof=1):.4f} | t = {t:+.2f} "
      f"(t_crit = {t_crit}) -> {'ЗНАЧИМО' if abs(t) > t_crit else 'не значимо'}")

diffs = []
for r in pair_rows:
    va = _correct_vec(cfg_baseline.output_dir, r["seed"])
    vb = _correct_vec(cfg_rl.output_dir, r["seed"])
    if va and vb:
        diffs += [y - x for x, y in zip(va, vb)]
p = float("nan")
if diffs:
    arr = np.array(diffs, float)
    rng = np.random.default_rng(42)
    boots = np.array([arr[rng.integers(0, len(arr), len(arr))].mean()
                      for _ in range(10000)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    print(f"бутстреп:  {arr.mean():+.4f} [CI {lo:+.4f}, {hi:+.4f}], "
          f"p={p:.4f}, n={len(arr)} пар")

print("\nval @ последняя эпоха (контроль вклада best-epoch отбора):")
for r in pair_rows:
    with open(f"{cfg_baseline.output_dir}_d16_seed{r['seed']}/val_metric_history.json") as f:
        hm = json.load(f)
    with open(f"{cfg_rl.output_dir}_d16_seed{r['seed']}/val_metric_history.json") as f:
        hr = json.load(f)
    print(f"  s{r['seed']}: MLE {hm['history'][-1]['value']:.3f} (best e{hm['best_epoch']:.0f})"
          f" vs RL {hr['history'][-1]['value']:.3f} (best e{hr['best_epoch']:.0f})")

t_ok = abs(t) > t_crit
p_ok = (not np.isnan(p)) and p < 0.05
print("\nВЕРДИКТ (урок §4.16: значимость = согласие t-критерия, бутстрепа")
print("и устойчивого направления по сидам):")
if t_ok and p_ok:
    print(f"  RL-only > MLE ПОДТВЕРЖДЁН на n={n}: t и бутстреп значимы.")
elif all(x > 0 for x in d):
    print(f"  Направление сохранилось ({sum(x > 0 for x in d)}/{n} в плюс), "
          f"но значимости на n={n} нет (t={t:+.2f} при {t_crit}, p={p:.3f}).")
else:
    print(f"  Направление НЕ воспроизвелось — эффект был шумом/хаосом.")

with open(_summary_b_path, "w") as f:
    json.dump({"rows": paired_d16b,
               "pair_rl_mle": {"n": n, "mean": float(np.mean(d)),
                               "std": float(np.std(d, ddof=1)),
                               "t": float(t), "t_crit": t_crit,
                               "p_bootstrap": float(p),
                               "per_seed": per_seed}}, f, indent=2)
print(f"\nИтог сохранён: {_summary_b_path}")


[Resume] сид 45 уже завершён
[Resume] сид 46 уже завершён

D1.6+D1.6b: RL-only − MLE-only, n=7 сидов (парный, общий старт)
per-seed: ['s42: +0.002', 's43: +0.008', 's44: +0.022', 's45: +0.004', 's46: +0.004', 's47: +0.020', 's48: -0.008']
разность: +0.0074 ± 0.0105 | t = +1.87 (t_crit = 2.447) -> не значимо
бутстреп:  +0.0074 [CI -0.0017, +0.0166], p=0.1168, n=3500 пар

val @ последняя эпоха (контроль вклада best-epoch отбора):
  s42: MLE 0.675 (best e4) vs RL 0.660 (best e4)
  s43: MLE 0.640 (best e3) vs RL 0.645 (best e4)
  s44: MLE 0.635 (best e3) vs RL 0.640 (best e4)
  s45: MLE 0.675 (best e3) vs RL 0.655 (best e3)
  s46: MLE 0.650 (best e3) vs RL 0.645 (best e3)
  s47: MLE 0.640 (best e3) vs RL 0.655 (best e4)
  s48: MLE 0.630 (best e3) vs RL 0.640 (best e3)

ВЕРДИКТ (урок §4.16: значимость = согласие t-критерия, бутстрепа
и устойчивого направления по сидам):
  Направление НЕ воспроизвелось — эффект был шумом/хаосом.

Итог сохранён: ./gen_dialogue_ed_full/summary_d1_6b.json


# D2: генерация ответов на EmpatheticDialogues — ядро темы

(emotion, situation) -> первая реплика-ответ. Контрастные пары диалоговой
семантики: **off-topic** (ответ другого диалога — уместность) и
**incoherent** (перестановка предложений — связность); контроль — маска
промпта (рецепт статьи) и MLE. Мотивация из D1: на классификации
содержательные негативы дублировали различение CE; в генерации CE такого
различения не выполняет — честный полигон для гипотезы темы.

**Порядок запуска:** ячейки сверху вниз (D1-ячейки при полном summary
срабатывают в режиме агрегации, GPU не занимают). Сначала смоук-тест
данных, затем zero-shot, затем матрикс D2.1 (~6 ч).

In [16]:
# ============================================================
# CELL D2-1: Конфиг D2 — response generation на ED
# ============================================================
class CFG_D2:
    # ---- Core
    seed = 42
    output_dir = "./gen_dialogue_ed_d2"
    gradient_checkpointing = False
    use_cache = True
    bf16 = True
    fp16 = False

    # ---- Model / PEFT (как D1.6 — проверенная стабильность)
    model_name = "Qwen/Qwen2.5-3B"
    trust_remote_code = True
    peft_kind = "prefix_tuning"
    peft_method = "prefix_tuning"
    num_virtual_tokens = 20
    prefix_projection = True
    init_selection_inits = 4
    init_selection_probe_steps = 200

    # ---- Task
    task_kind = "response_gen"
    dataset_repo = "empathetic_dialogues"
    dataset_revision = "refs/convert/parquet"
    train_size = 6000
    val_size = 200
    test_size = 500
    max_source_len = 128
    max_target_len = 48          # с EOS (генерация — не cls)
    max_total_len = max_source_len + max_target_len
    prompt_template = "Emotion: {emotion}\nSituation: {source}\nResponse:"
    padding_side = "left"
    eval_max_new_tokens = 48
    gen_max_new_tokens = 48
    eval_metrics = ["rouge1", "rouge2", "rougeL", "distinct1", "distinct2"]

    # ---- Schedule (одна фаза, контраст с 1-й эпохи — урок §4.6)
    phase1_epochs = 4
    total_epochs = 4
    batch_size = 8
    eval_batch_size = 8
    gradient_accumulation_steps = 1
    learning_rate = 5e-4
    phase2_lr = 1e-4
    weight_decay = 0.0
    warmup_steps = 200
    lr_scheduler = "cosine"
    max_grad_norm = 1.0
    logging_steps = 10
    eval_strategy = "epoch"
    save_strategy = "epoch"
    save_total_limit = 1
    load_best_model_at_end = False
    remove_unused_columns = False
    dataloader_pin_memory = True
    report_to = "none"

    # ---- Contrast (мягкие значения D1.6: стабильны 15/15)
    alpha = 1.0
    beta = 0.1
    contrast_mode = "offtopic"
    contrast_from_start = True
    contrastive_dropout = 0.1
    contrastive_tau = 0.5
    k_negatives = 1
    contrast_neg_in_graph = False   # D2.3: градиент через негатив (абляция)
    reward_baseline_beta = 0.9      # D2.2b: инерция EMA-бейзлайна REINFORCE

    # ---- RL: в D2.1 ВЫКЛЮЧЕН — сначала ядро темы (контраст);
    # RL вернём в D2.2 (reward = ROUGE-L) после чистого ответа на гипотезу
    gamma = 0.0
    gamma_auto = False
    gamma_target_frac = 0.3
    sigma = 0.02
    rl_interval = 50
    rl_num_batches = 16
    rl_subset_size = 600
    use_reward_baseline = True
    gen_do_sample = False
    gen_temperature = 0.7
    gen_top_p = 0.9
    reward_metric = "rougeL"

    divergence_ce_threshold = 8.0
    guard_start_step = 100
    val_rouge_examples = 200


cfg2 = CFG_D2()
assert cfg2.task_kind == "response_gen"
assert cfg2.gamma == 0.0 and not cfg2.gamma_auto
print("Config D2 OK (response_gen, RL выключен до D2.2)")
print(f"  model: {cfg2.model_name} | src<={cfg2.max_source_len} tgt<={cfg2.max_target_len} (с EOS)")
print(f"  шаблон: {cfg2.prompt_template!r}")
print(f"  contrast: mode={cfg2.contrast_mode}, beta={cfg2.beta}, tau={cfg2.contrastive_tau}")

Config D2 OK (response_gen, RL выключен до D2.2)
  model: Qwen/Qwen2.5-3B | src<=128 tgt<=48 (с EOS)
  шаблон: 'Emotion: {emotion}\nSituation: {source}\nResponse:'
  contrast: mode=offtopic, beta=0.1, tau=0.5


In [17]:
# ============================================================
# CELL D2-2: Данные ED (диалоги) + диалоговые негативы
# Эпизод = (эмоция, ситуация, ПЕРВАЯ реплика по utterance_idx).
# Негативы (детерминированы RNG от имени сплита, не от сида):
#   offtopic   — реплика ДРУГОГО эпизода (неуместность);
#   incoherent — перестановка предложений золотой реплики (связность),
#                фолбэк для коротких — перестановка слов; вырожденные
#                (негатив == позитив) заменяются на offtopic.
# ============================================================
import re as _re
import random as _random

SENT_SPLIT = _re.compile(r"(?<=[.!?])\s+")


def load_ed_conversations(split):
    """ED -> [(emotion, situation, первая реплика ДРУГОГО спикера)].

    Особенность ED: utterance_idx 0 — пересказ ситуации её автором
    (тем же спикером), а не ответ. Эмпатичный ответ = первая реплика
    другого speaker_idx (обычно idx 1-2). Без второго спикера 64/17844
    диалогов — пропускаем."""
    files = list_repo_files(cfg2.dataset_repo, repo_type="dataset",
                            revision=cfg2.dataset_revision)
    names = sorted(f for f in files
                   if f.startswith(f"default/{split}/") and f.endswith(".parquet"))
    assert names, f"нет parquet-шардов для split={split}"
    paths = [hf_hub_download(cfg2.dataset_repo, n, repo_type="dataset",
                             revision=cfg2.dataset_revision) for n in names]
    ds = load_dataset("parquet", data_files=paths, split="train")
    by_conv = {}
    for r in ds:
        by_conv.setdefault(r["conv_id"], []).append(r)
    convs = []
    for cid in sorted(by_conv):
        rows = sorted(by_conv[cid], key=lambda r: r["utterance_idx"])
        spk0 = rows[0]["speaker_idx"]
        reply = next((r for r in rows if r["speaker_idx"] != spk0), None)
        if reply is None:
            continue
        utt = reply["utterance"].replace("_comma_", ",").strip()
        sit = rows[0]["prompt"].strip()
        emo = rows[0]["context"].strip()
        if utt and sit and emo:
            convs.append({"emotion": emo, "situation": sit, "response": utt})
    return convs


def make_incoherent(text, rng):
    """Негатив-перестановка: предложения (2 = детерминированный свап,
    >2 = shuffle с retry); для 1-предложечных — слова (от 2 слов, с
    retry); None = вырожден (одно слово/одно предложение из 1 слова)."""
    sents = [s for s in SENT_SPLIT.split(text.strip()) if s]
    if len(sents) == 2:
        return f"{sents[1]} {sents[0]}".strip()
    if len(sents) > 2:
        for _ in range(3):
            rng.shuffle(sents)
            cand = " ".join(sents).strip()
            if cand != text.strip():
                return cand
    words = text.split()
    if len(words) >= 2:
        for _ in range(3):
            rng.shuffle(words)
            cand = " ".join(words)
            if cand != text.strip():
                return cand
    return None


def attach_negatives(rows, split, k_off=4):
    """neg_off (СОВМЕСТИМО с D2.1: j=perm[i], j!=i), neg_off2..k_off —
    дополнительные различные чужие ответы (D2.3: k негативов для
    Σ_j из Eq. 3), neg_inc — перестановка."""
    rng_off = _random.Random(f"offtopic-{split}")
    perm = list(range(len(rows)))
    rng_off.shuffle(perm)
    rng_inc = _random.Random(f"incoherent-{split}")
    n_degen = 0
    n = len(rows)
    for i, c in enumerate(rows):
        used = {i}
        j = perm[i]
        if j == i:
            j = (j + 1) % n
        used.add(j)
        c["neg_off"] = rows[j]["response"]
        extra, step = 0, 1
        while extra < k_off - 1 and step < n:
            cand = perm[(i + step) % n]
            if cand == i:
                cand = (cand + 1) % n
            if cand not in used:
                used.add(cand)
                c[f"neg_off{extra + 2}"] = rows[cand]["response"]
                extra += 1
            step += 1
        while extra < k_off - 1:      # датасет мал — дублируем (не бывает)
            c[f"neg_off{extra + 2}"] = c["neg_off"]
            extra += 1
        inc = make_incoherent(c["response"], rng_inc)
        if inc is None or inc.strip() == c["response"].strip():
            inc = c["neg_off"]      # вырожденный негатив -> offtopic
            n_degen += 1
        c["neg_inc"] = inc
    return n_degen


def prepare_split(convs_list, name):
    rows = [dict(r) for r in convs_list]
    ndeg = attach_negatives(rows, name)
    return rows, ndeg


ed2_train_full = load_ed_conversations("train")
ed2_val_full = load_ed_conversations("validation")
ed2_test_full = load_ed_conversations("test")
print(f"ED dialogues: train {len(ed2_train_full)} | "
      f"val {len(ed2_val_full)} | test {len(ed2_test_full)}")
assert cfg2.rl_subset_size + cfg2.train_size <= len(ed2_train_full)

rows_rl,    deg_rl    = prepare_split(ed2_train_full[:cfg2.rl_subset_size], "rl")
rows_train, deg_train = prepare_split(
    ed2_train_full[cfg2.rl_subset_size:cfg2.rl_subset_size + cfg2.train_size], "train")
rows_val,   deg_val   = prepare_split(ed2_val_full[:cfg2.val_size], "val")
rows_test,  deg_test  = prepare_split(ed2_test_full[:cfg2.test_size], "test")
print(f"Вырожденных (incoherent->offtopic) негативов: "
      f"train {deg_train}/{len(rows_train)}, val {deg_val}, test {deg_test}")

raw2_rl    = HFDataset.from_list(rows_rl)
raw2_train = HFDataset.from_list(rows_train)
raw2_val   = HFDataset.from_list(rows_val)
raw2_test  = HFDataset.from_list(rows_test)


NEG_FIELD_TO_PFX = {"neg_off": "negoff", "neg_off2": "negoff2",
                    "neg_off3": "negoff3", "neg_off4": "negoff4",
                    "neg_inc": "neginc"}


def tokenize_d2(examples):
    neg_fields = [f for f in NEG_FIELD_TO_PFX if f in examples]
    pfxs = [""] + [NEG_FIELD_TO_PFX[f] for f in neg_fields]
    # пустой префикс -> "input_ids"; негативный -> "negoff_input_ids"
    # (баг было: f"{p}{k}" без "_" для негативных префиксов)
    out = {}
    for p in pfxs:
        for k in ("input_ids", "attention_mask", "labels"):
            out[k if p == "" else f"{p}_{k}"] = []
    max_total = cfg2.max_total_len

    def enc_target(text):
        ids = tok(" " + text, add_special_tokens=False, truncation=True,
                  max_length=cfg2.max_target_len - 1)["input_ids"]
        return ids + [tok.eos_token_id]

    base = zip(examples["emotion"], examples["situation"], examples["response"])
    for bi, (emo, sit, resp) in enumerate(base):
        src = cfg2.prompt_template.format(emotion=emo, source=sit)
        src_ids = tok(src, add_special_tokens=False, truncation=True,
                      max_length=cfg2.max_source_len)["input_ids"]
        if not src_ids:
            src_ids = [tok.pad_token_id]

        tgt_ids = enc_target(resp)
        if len(src_ids) + len(tgt_ids) > max_total:
            src_ids = src_ids[:max(1, max_total - len(tgt_ids))]
        out["input_ids"].append(src_ids + tgt_ids)
        out["attention_mask"].append([1] * (len(src_ids) + len(tgt_ids)))
        out["labels"].append([-100] * len(src_ids) + tgt_ids)

        for f in neg_fields:
            pfx = NEG_FIELD_TO_PFX[f]
            n_ids = enc_target(examples[f][bi])
            if len(src_ids) + len(n_ids) > max_total:
                n_ids = n_ids[:max(1, max_total - len(src_ids))]
            out[f"{pfx}_input_ids"].append(src_ids + n_ids)
            out[f"{pfx}_attention_mask"].append([1] * (len(src_ids) + len(n_ids)))
            out[f"{pfx}_labels"].append([-100] * len(src_ids) + n_ids)
    return out


train2_ds = raw2_train.map(tokenize_d2, batched=True, remove_columns=raw2_train.column_names)
val2_ds   = raw2_val.map(tokenize_d2, batched=True, remove_columns=raw2_val.column_names)
test2_ds  = raw2_test.map(tokenize_d2, batched=True, remove_columns=raw2_test.column_names)
rl2_ds    = raw2_rl.map(tokenize_d2, batched=True, remove_columns=raw2_rl.column_names)
rl2_ds = rl2_ds.add_column("ref_idx", list(range(len(rl2_ds))))

rl2_references   = raw2_rl["response"]
val2_references  = raw2_val["response"]
test2_references = raw2_test["response"]

for d in (train2_ds, val2_ds, test2_ds, rl2_ds):
    d.set_format("torch")

# ВАЖНО: перезапривязываем глобальные имена датасетов к D2 —
# run_experiment / prepare_trainer / probe работают с ними.
# D1-агрегация читает только JSON с диска, конфликта нет.
train_ds, val_ds, test_ds, rl_ds = train2_ds, val2_ds, test2_ds, rl2_ds
rl_references, val_references, test_references = (
    rl2_references, val2_references, test2_references)

print(f"D2: train {len(train2_ds)} | val {len(val2_ds)} | "
      f"test {len(test2_ds)} | RL hold-out {len(rl2_ds)} (для D2.2)")
assert (train2_ds[0]["labels"] != -100).sum() > 0
assert train2_ds[0]["labels"][-1].item() == tok.eos_token_id, \
    "EOS обязан быть последним токеном таргета (генерация)"
print("Инварианты OK: супервизируемые токены есть, EOS в конце")

ED dialogues: train 17780 | val 2758 | test 2540
Вырожденных (incoherent->offtopic) негативов: train 41/6000, val 0, test 1


Map: 100%|██████████| 600/600 [00:00<00:00, 2124.29 examples/s]

D2: train 6000 | val 200 | test 500 | RL hold-out 600 (для D2.2)
Инварианты OK: супервизируемые токены есть, EOS в конце


In [18]:
# ============================================================
# CELL D2-3: СМОУК-ТЕСТ данных D2 (запустить ДО матрикса)
# ============================================================
_tgt_tok, _no_eos = [], 0
for i in range(300):
    labs = train2_ds[i]["labels"].tolist()
    _tgt_tok.append(sum(1 for x in labs if x != -100))
    if labs[-1] != tok.eos_token_id:
        _no_eos += 1
print(f"таргет-токены (300 прим.): mean {np.mean(_tgt_tok):.1f} | "
      f"p95 {np.percentile(_tgt_tok, 95):.0f} | max {max(_tgt_tok)} | без EOS: {_no_eos}")

same_off = sum(1 for r in rows_train[:300] if r["neg_off"].strip() == r["response"].strip())
same_inc = sum(1 for r in rows_train[:300] if r["neg_inc"].strip() == r["response"].strip())
multi = sum(1 for r in rows_train[:300] if len([s for s in SENT_SPLIT.split(r["response"]) if s]) >= 2)
print(f"негатив == позитив: offtopic {same_off}/300, incoherent {same_inc}/300")
print(f"многопредложечных ответов: {multi}/300 "
      f"(остальным incoherent = перестановка слов)")

src_words = np.mean([len(r["situation"].split()) for r in rows_train[:300]])
resp_words = np.mean([len(r["response"].split()) for r in rows_train[:300]])
print(f"слова: ситуация {src_words:.1f} | ответ {resp_words:.1f} "
      f"(проверка max_source_len/max_target_len)")

# ИНВАРИАНТ КОЛЛАТОРА: легаси-конвенция 0/0 (см. комментарий в
# коллаторе и §6.3): маска только {0,1}, паддинг input_ids = 0
_bs = [train2_ds[i] for i in range(8)]
_b = causal_lm_collator(_bs)
assert set(_b["attention_mask"].unique().tolist()) <= {0, 1}, \
    "attention_mask содержит не-{0,1}: коллатор сломан!"
_padded_rows = (_b["attention_mask"].sum(1) < _b["attention_mask"].size(1))
if _padded_rows.any():
    _r = int(_padded_rows.nonzero()[0])
    _n_pad = int((_b["attention_mask"][_r] == 0).sum())
    assert (_b["input_ids"][_r][:_n_pad] == 0).all(), \
        "input_ids на левом паддинге != 0 (легаси-конвенция)"
print("Инвариант коллатора OK: маска {0,1}, паддинг input_ids = 0 (легаси)")

print("\nПримеры (POS = золотой ответ, OFF = чужой, INC = переставленный):")
for r in rows_train[:4]:
    print("-" * 80)
    print(f"[{r['emotion']}] {r['situation'][:100]!r}")
    print(f"  POS: {r['response'][:110]!r}")
    print(f"  OFF: {r['neg_off'][:110]!r}")
    print(f"  INC: {r['neg_inc'][:110]!r}")

таргет-токены (300 прим.): mean 14.6 | p95 27 | max 45 | без EOS: 0
негатив == позитив: offtopic 0/300, incoherent 0/300
многопредложечных ответов: 179/300 (остальным incoherent = перестановка слов)
слова: ситуация 20.9 | ответ 11.0 (проверка max_source_len/max_target_len)
Инвариант коллатора OK: маска {0,1}, паддинг input_ids = 0 (легаси)

Примеры (POS = золотой ответ, OFF = чужой, INC = переставленный):
--------------------------------------------------------------------------------
[disgusted] "I was babysitting for someone I didn't know very well. When they brought the baby in the car seat_co"
  POS: "Ew, that's really disturbing.  The sight of cockroaches always gives me the willies."
  OFF: "OMG!  That's insane.  How did you react?"
  INC: "The sight of cockroaches always gives me the willies. Ew, that's really disturbing."
--------------------------------------------------------------------------------
[anxious] "I've going through a spell as of late when I've been pacing back a

In [19]:
# ============================================================
# CELL D2-4: метрики генерации — Distinct-n (+ BERTScore опционально)
# ============================================================
def distinct_n(texts):
    """Distinct-1/2: доля уникальных уни-/биграмм по корпусу генераций."""
    unis, bis = set(), set()
    ntok, npairs = 0, 0
    for t in texts:
        w = t.split()
        unis.update(w)
        bis.update(zip(w, w[1:]))
        ntok += len(w)
        npairs += max(len(w) - 1, 0)
    return len(unis) / max(ntok, 1), len(bis) / max(npairs, 1)

try:
    from bert_score import score as _bert_score_fn
    HAVE_BERTSCORE = True
    print("bert-score установлен: BERTScore можно считать по predictions")
except Exception:
    HAVE_BERTSCORE = False
    print("bert-score НЕ установлен (опционально: pip install bert-score); "
          "BERTScore считается отдельной ячейкой по сохранённым predictions")
print("distinct_n готов")

bert-score установлен: BERTScore можно считать по predictions
distinct_n готов


In [20]:
# ============================================================
# CELL D2-5: конфигурации D2.1 — ядро темы
# ============================================================
class D2_MLE(CFG_D2):
    """Базлайн: чистое CE-дообучение промпта."""
    output_dir = "./gen_d2_mle"
    beta = 0.0


class D2_Mask(D2_MLE):
    """Контроль: маска промпта (рецепт статьи), мягкие значения."""
    output_dir = "./gen_d2_mask"
    beta = 0.1
    contrast_mode = "mask"


class D2_Offtopic(D2_MLE):
    """Ядро темы: негатив = (контекст + ответ ДРУГОГО диалога)."""
    output_dir = "./gen_d2_offtopic"
    beta = 0.1
    contrast_mode = "offtopic"


class D2_Incoherent(D2_MLE):
    """Ядро темы: негатив = (контекст + переставленный золотой ответ)."""
    output_dir = "./gen_d2_incoherent"
    beta = 0.1
    contrast_mode = "incoherent"


cfg2_baseline = D2_MLE()
cfg2_mask = D2_Mask()
cfg2_off = D2_Offtopic()
cfg2_inc = D2_Incoherent()

print("=" * 78)
print("D2.1 CONFIGS (response_gen, Qwen2.5-3B, чистый eval, общий старт)")
print("=" * 78)
for c, name in [(cfg2_baseline, "MLE-only"),
                (cfg2_mask, "contrast-mask (контроль статьи)"),
                (cfg2_off, "contrast-offtopic (ядро)"),
                (cfg2_inc, "contrast-incoherent (ядро)")]:
    print(f"{name:<34} mode={c.contrast_mode:<12} beta={c.beta} -> {c.output_dir}")
print("=" * 78)

D2.1 CONFIGS (response_gen, Qwen2.5-3B, чистый eval, общий старт)
MLE-only                           mode=offtopic     beta=0.0 -> ./gen_d2_mle
contrast-mask (контроль статьи)    mode=mask         beta=0.1 -> ./gen_d2_mask
contrast-offtopic (ядро)           mode=offtopic     beta=0.1 -> ./gen_d2_offtopic
contrast-incoherent (ядро)         mode=incoherent   beta=0.1 -> ./gen_d2_incoherent


In [21]:
# ============================================================
# CELL D2-6: zero-shot референс D2 (базовая модель, тот же шаблон)
# ============================================================
gc.collect()
torch.cuda.empty_cache()
set_seed(cfg2.seed)

zs2_base = AutoModelForCausalLM.from_pretrained(
    cfg2.model_name, dtype=torch.bfloat16, trust_remote_code=True)
zs2_base.eval().to("cuda" if torch.cuda.is_available() else "cpu")

prompts2 = [cfg2.prompt_template.format(emotion=raw2_test[i]["emotion"],
                                        source=raw2_test[i]["situation"])
            for i in range(len(raw2_test))]

zs2_preds = []
with torch.no_grad():
    for i in range(0, len(prompts2), 8):
        enc = tok(prompts2[i:i + 8], return_tensors="pt", padding=True,
                  truncation=True, max_length=cfg2.max_source_len + 8).to(zs2_base.device)
        out = zs2_base.generate(
            **enc, max_new_tokens=cfg2.eval_max_new_tokens, do_sample=False,
            num_beams=1, pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for j in range(enc["input_ids"].size(0)):
            zs2_preds.append(tok.decode(
                out[j, enc["input_ids"].size(1):], skip_special_tokens=True).strip() or " ")

del zs2_base
gc.collect()
torch.cuda.empty_cache()

zs2_rouge = evaluate.load("rouge")
zs2_scores = zs2_rouge.compute(predictions=zs2_preds, references=test2_references,
                               rouge_types=["rouge1", "rouge2", "rougeL"])
zs2_scores["distinct1"], zs2_scores["distinct2"] = distinct_n(zs2_preds)

print("=" * 70)
print("ZERO-SHOT D2: генерация ответа (ED, test 500, greedy 48)")
print("=" * 70)
for k, v in zs2_scores.items():
    print(f"{k}: {v:.4f}")

os.makedirs(cfg2.output_dir, exist_ok=True)
with open(f"{cfg2.output_dir}/zeroshot_d2.json", "w") as f:
    json.dump({"experiment": "zero-shot-d2", "scores": zs2_scores}, f, indent=2)

print("\nПримеры (REF | GEN):")
for i in range(4):
    print("-" * 80)
    print(f"[{raw2_test[i]['emotion']}] {raw2_test[i]['situation'][:90]!r}")
    print(f"  REF: {test2_references[i][:100]!r}")
    print(f"  GEN: {zs2_preds[i][:100]!r}")

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10552.69it/s]


ZERO-SHOT D2: генерация ответа (ED, test 500, greedy 48)
rouge1: 0.0931
rouge2: 0.0097
rougeL: 0.0809
distinct1: 0.1207
distinct2: 0.3853

Примеры (REF | GEN):
--------------------------------------------------------------------------------
[guilty] 'I felt guilty when I was driving home one night and a person tried to fly into my lane_com'
  REF: 'Did you suffer any injuries?'
  GEN: 'I was so scared that I almost drove off the road. I was so scared that I almost drove off the road. '
--------------------------------------------------------------------------------
[surprised] 'I was in for a treat the other day when my husband got home with some chocolate. He likes '
  REF: 'Wow that must have been a surprise for you'
  GEN: 'I was surprised. I was not expecting it. I was not expecting him to get home with chocolate. I was n'
--------------------------------------------------------------------------------
[trusting] 'I have problems with the city and their constant tax increase on my 

In [22]:
# ============================================================
# ЗАПУСК D2.1 (RESUME-SAFE): 7 сидов x {MLE, mask, offtopic, incoherent}
# Общий старт на сид (probe -> shared_init.pt на диск), парный дизайн.
# Время: MLE ~14 мин, контраст ~20 мин, probe ~5 -> ~80 мин/сид;
# 4 новых сида (45-48) -> ~5.5 ч. Прогон 42-44 уже завершён — resume.
# Статистика: парный t + бутстреп по per-example ROUGE-L + траектории.
# ============================================================
import copy
from math import sqrt

# 7 сидов (расширение для мощности, все новые сиды без отбора)
SEEDS2 = [42, 43, 44, 45, 46, 47, 48]
D21_CONFIGS = [
    (cfg2_mask, "contrast-mask"),
    (cfg2_off, "contrast-offtopic"),
    (cfg2_inc, "contrast-incoherent"),
]
NEEDED2 = {"seed", "MLE-only", "contrast-mask", "contrast-offtopic",
           "contrast-incoherent"}

dirs2 = {"MLE-only": cfg2_baseline.output_dir,
         "contrast-mask": cfg2_mask.output_dir,
         "contrast-offtopic": cfg2_off.output_dir,
         "contrast-incoherent": cfg2_inc.output_dir}

_summary2_path = f"{cfg2.output_dir}/summary_d2_1.json"
paired_d21 = []
if os.path.exists(_summary2_path):
    with open(_summary2_path) as f:
        paired_d21 = json.load(f).get("rows", [])

done2 = {r["seed"] for r in paired_d21 if NEEDED2.issubset(r)}
todo2 = [s for s in SEEDS2 if s not in done2]
print(f"D2.1 resume: готово {sorted(done2)}, к запуску {todo2}")

for seed in todo2:
    c_mle = copy.copy(cfg2_baseline)
    c_mle.seed = seed
    c_mle.output_dir = f"{cfg2_baseline.output_dir}_d21_seed{seed}"
    shared_path = os.path.join(c_mle.output_dir, "shared_init.pt")
    _mle_json = os.path.join(c_mle.output_dir, "results.json")
    if os.path.exists(shared_path) and os.path.exists(_mle_json):
        shared = torch.load(shared_path, map_location="cpu")
        with open(_mle_json) as f:
            scores_mle = json.load(f)["test_scores"]
        print(f"[Resume] сид {seed}: общий старт и MLE с диска")
    else:
        scores_mle, shared = run_experiment(
            c_mle, f"MLE-only D2.1 (seed {seed})", return_init_state=True)
        os.makedirs(c_mle.output_dir, exist_ok=True)
        torch.save(shared, shared_path)
    assert shared is not None

    row = {"seed": seed, "MLE-only": scores_mle["rougeL"],
           "MLE-only-r1": scores_mle["rouge1"],
           "MLE-only-d1": scores_mle["distinct1"],
           "MLE-only-d2m": scores_mle["distinct2"]}

    for base_cfg, name in D21_CONFIGS:
        c = copy.copy(base_cfg)
        c.seed = seed
        c.output_dir = f"{base_cfg.output_dir}_d21_seed{seed}"
        s = run_experiment(c, f"{name} D2.1 (seed {seed}, общий старт)",
                           init_state=shared)
        row[name] = s["rougeL"]
        row[f"{name}-r1"] = s["rouge1"]
        row[f"{name}-d1"] = s["distinct1"]
        row[f"{name}-d2m"] = s["distinct2"]

    paired_d21 = [r for r in paired_d21 if r["seed"] != seed] + [row]
    del shared
    gc.collect()
    torch.cuda.empty_cache()
    with open(_summary2_path, "w") as f:
        json.dump({"rows": paired_d21}, f, indent=2)
    print(f"\n>>> Сид {seed} завершён (rougeL): "
          f"{ {k: round(v, 4) for k, v in row.items() if '-r' not in k and '-d' not in k} }\n")

# ============================================================
# АГРЕГАЦИЯ D2.1
# ============================================================
names2 = ["MLE-only"] + [n for _, n in D21_CONFIGS]

print("\n" + "=" * 92)
print(f"D2.1: генерация ответов ED, чистый протокол, {len(paired_d21)} сидов, общий старт")
print("=" * 92)
print(f"{'Конфиг':<24}{'ROUGE-L':<12}{'std':<9}{'R-1':<8}{'Dist-1':<8}{'Dist-2':<8}{'per-seed (R-L)'}")
print("-" * 92)
for n in names2:
    v = [r[n] for r in paired_d21]
    r1 = np.mean([r[f"{n}-r1"] for r in paired_d21])
    d1 = np.mean([r[f"{n}-d1"] for r in paired_d21])
    d2m = np.mean([r[f"{n}-d2m"] for r in paired_d21])
    print(f"{n:<24}{np.mean(v):<12.4f}{np.std(v):<9.4f}{r1:<8.4f}{d1:<8.4f}{d2m:<8.4f}"
          f"{[round(x, 4) for x in v]}")
print("-" * 92)
if "zs2_scores" in globals():
    print(f"zero-shot: rougeL={zs2_scores['rougeL']:.4f} "
          f"distinct1={zs2_scores['distinct1']:.4f}")
print("=" * 92)

t_crit = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365}

def load_rouge_vec(config_name, seed):
    p = f"{dirs2[config_name]}_d21_seed{seed}/results.json"
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f).get("per_example_rougeL")
    return None

print("\nПарные разности (ROUGE-L):")
print("а) t-критерий по сидам (df = n-1):")
pairs2 = [("contrast-mask", "MLE-only"),
          ("contrast-offtopic", "MLE-only"),
          ("contrast-incoherent", "MLE-only"),
          ("contrast-offtopic", "contrast-mask"),
          ("contrast-incoherent", "contrast-mask"),
          ("contrast-incoherent", "contrast-offtopic")]
for a, b in pairs2:
    d = [r[a] - r[b] for r in paired_d21]
    se = np.std(d, ddof=1) / sqrt(len(d)) if len(d) > 1 else float("nan")
    t = np.mean(d) / se if se and se > 0 else float("nan")
    print(f"  {a:<22} − {b:<14}: {np.mean(d):+.4f} ± {np.std(d):.4f} | "
          f"t={t:+.2f} | per-seed {[f'{x:+.4f}' for x in d]}")

print("\nб) парный бутстреп по примерам (per-example ROUGE-L, 10000):")
for a, b in pairs2:
    diffs = []
    for r in paired_d21:
        va, vb = load_rouge_vec(a, r["seed"]), load_rouge_vec(b, r["seed"])
        if va and vb:
            diffs += [x - y for x, y in zip(va, vb)]
    if not diffs:
        print(f"  {a} − {b}: per_example_rougeL не найден")
        continue
    arr = np.array(diffs, float)
    rng = np.random.default_rng(42)
    boots = np.array([arr[rng.integers(0, len(arr), len(arr))].mean()
                      for _ in range(10000)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    print(f"  {a:<22} − {b:<14}: {arr.mean():+.4f} "
          f"[CI {lo:+.4f}, {hi:+.4f}], p={p:.4f} "
          f"({'ЗНАЧИМО' if p < 0.05 else 'н.з.'}), n={len(arr)}")

def load_hist2(config_name, seed):
    p = f"{dirs2[config_name]}_d21_seed{seed}/val_metric_history.json"
    if os.path.exists(p):
        with open(p) as f:
            return [x["value"] for x in json.load(f)["history"]]
    return None

print("\nв) Траектории val rougeL по эпохам:")
for n in names2:
    for r in paired_d21:
        h = load_hist2(n, r["seed"])
        if h:
            print(f"  {n:<24}s{r['seed']}: {' -> '.join(f'{v:.3f}' for v in h)}")

print("\nг) Разности на ФИКСИРОВАННОЙ последней эпохе:")
for a, b in pairs2:
    dl = [load_hist2(a, r["seed"])[-1] - load_hist2(b, r["seed"])[-1]
          for r in paired_d21
          if load_hist2(a, r["seed"]) and load_hist2(b, r["seed"])]
    if dl:
        print(f"  {a:<22} − {b:<14} @last: {np.mean(dl):+.4f} | "
              f"per-seed {[f'{x:+.4f}' for x in dl]} | "
              f"{sum(x > 0 for x in dl)}/{len(dl)} в плюс")

final_d21 = {
    "per_seed": paired_d21,
    "mean": {n: float(np.mean([r[n] for r in paired_d21])) for n in names2},
    "std": {n: float(np.std([r[n] for r in paired_d21])) for n in names2},
}
with open(_summary2_path, "w") as f:
    json.dump(final_d21, f, indent=2)
print(f"\nSummary сохранён: {_summary2_path}")
print("Урок §4.16: значимость = согласие t + бутстрепа + репликация направления.")

D2.1 resume: готово [], к запуску [42, 43, 44, 45, 46, 47, 48]
[Resume] сид 42: общий старт и MLE с диска

STARTING EXPERIMENT: contrast-mask D2.1 (seed 42, общий старт)



Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10080.95it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.1
[Step 0] CE/token (EMA): 1.981 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,2.326377,2.386463
2,2.452921,2.378219
3,2.368140,2.374088
4,2.224291,2.390268


[Step 10] CE/token (EMA): 2.076 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.122 | Contrast[mask]: 0.692 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.168 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.262 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.228 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.257 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.237 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.255 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.249 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 2.275 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 2.281 | Contrast[mask]: 0.693 | RL: 0.000 | Reward: 0.0000
[Step 120] CE/token (EMA): 2.278 | Contrast[mask]: 0.693 | RL: 

...

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9856.18it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=incoherent, from_start=True, beta=0.1
[Step 0] CE/token (EMA): 1.981 | Contrast[incoherent]: 0.687 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,4.714479,4.686142
2,4.249590,4.151352


[Step 10] CE/token (EMA): 2.076 | Contrast[incoherent]: 0.678 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.122 | Contrast[incoherent]: 0.688 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.168 | Contrast[incoherent]: 0.689 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.262 | Contrast[incoherent]: 0.687 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.227 | Contrast[incoherent]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.256 | Contrast[incoherent]: 0.689 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.235 | Contrast[incoherent]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.253 | Contrast[incoherent]: 0.687 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.247 | Contrast[incoherent]: 0.687 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 2.274 | Contrast[incoherent]: 0.686 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 2.281 | Contrast[incoherent]: 0.683 | RL: 0.000 | Reward: 0.00

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL D2-8: ЗАПУСК D2.2 — RL-only с reward = ROUGE-L (RESUME-SAFE)
# Единственная положительная нить across задач (D1: RL−MLE 5/5 в плюс).
# Пары с УЖЕ существующими MLE-прогонами D2.1: shared_init.pt на диске
# (probe не повторяем), добираем только RL-прогоны.
# RL-бюджет ужат под генерацию: событие каждые 100 шагов по 64
# примерам — reward-генерация 48 токенов в ~8 раз дороже D1;
# 64 примера/событие = нижняя граница рекомендации критика.
# Время: ~30 мин/сид (14 MLE-эквивалент + ~15 мин RL-событий) -> ~1.5 ч.
# ВАЖНО (внесено): compute_generation_reward теперь принимает task_cfg
# — reward считается по КОНФИГУ ЗАДАЧИ (ROUGE-L, 48 токенов), а не по
# глобальному cfg (который остаётся D1-классификацией).
# ============================================================
import copy
from math import sqrt

# 7 сидов (расширение для мощности; MLE-пары берутся из D2.1)
SEEDS_D22 = [42, 43, 44, 45, 46, 47, 48]


class D2_RL_Only(CFG_D2):
    """D2.2: только REINFORCE, reward = ROUGE-L на чистом входе
    (strip_label_tokens -> greedy generate -> rougeL против референса
    из RL hold-out 600). γ auto-калибруется по первому событию."""
    output_dir = "./gen_d2_rl_only"

    beta = 0.0
    contrast_mode = "mask"     # не используется при β=0
    gamma = 2e-5               # стартовое; auto-калибровка перепишет
    gamma_auto = True
    rl_interval = 100          # было 50: генерационный reward дороже
    rl_num_batches = 8         # 64 примера/событие (критик: 64+)


cfg2_rl = D2_RL_Only()
assert cfg2_rl.gamma_auto and cfg2_rl.beta == 0.0
print(f"D2.2: RL-only | reward=rougeL | событие каждые {cfg2_rl.rl_interval} шагов "
      f"по {cfg2_rl.rl_num_batches * cfg2_rl.batch_size} примерам")

_summary_d22_path = f"{cfg2.output_dir}/summary_d2_2.json"
paired_d22 = []
if os.path.exists(_summary_d22_path):
    with open(_summary_d22_path) as f:
        paired_d22 = json.load(f).get("rows", [])

for seed in SEEDS_D22:
    if any(r["seed"] == seed for r in paired_d22):
        print(f"[Resume] сид {seed} уже завершён")
        continue

    # MLE и общий старт — из прогонов D2.1 (не трогаем, не перезапускаем)
    mle_dir = f"{cfg2_baseline.output_dir}_d21_seed{seed}"
    mle_json = os.path.join(mle_dir, "results.json")
    shared_path = os.path.join(mle_dir, "shared_init.pt")
    assert os.path.exists(mle_json) and os.path.exists(shared_path), \
        f"нет MLE D2.1 / общего старта для сида {seed}: {mle_dir}"

    with open(mle_json) as f:
        scores_mle = json.load(f)["test_scores"]
    shared = torch.load(shared_path, map_location="cpu")
    print(f"[Paired] сид {seed}: MLE D2.1 rougeL={scores_mle['rougeL']:.4f}, "
          f"общий старт с диска")

    c_rl = copy.copy(cfg2_rl)
    c_rl.seed = seed
    c_rl.output_dir = f"{cfg2_rl.output_dir}_d22_seed{seed}"
    rl_json = os.path.join(c_rl.output_dir, "results.json")
    if os.path.exists(rl_json):
        with open(rl_json) as f:
            scores_rl = json.load(f)["test_scores"]
        print(f"[Resume] сид {seed}: RL-only с диска")
    else:
        scores_rl = run_experiment(
            c_rl, f"RL-only D2.2 (seed {seed}, общий старт D2.1)",
            init_state=shared)

    row = {"seed": seed,
           "MLE-only": scores_mle["rougeL"],
           "MLE-only-d1": scores_mle["distinct1"],
           "RL-only": scores_rl["rougeL"],
           "RL-only-d1": scores_rl["distinct1"]}
    paired_d22 = [r for r in paired_d22 if r["seed"] != seed] + [row]

    del shared
    gc.collect()
    torch.cuda.empty_cache()
    with open(_summary_d22_path, "w") as f:
        json.dump({"rows": paired_d22}, f, indent=2)
    print(f">>> Сид {seed} завершён: rougeL MLE={row['MLE-only']:.4f} "
          f"RL={row['RL-only']:.4f} (diff {row['RL-only']-row['MLE-only']:+.4f})\n")

# ============================================================
# АГРЕГАЦИЯ D2.2: RL-only против MLE (D2.1) на общем старте
# ============================================================
print("\n" + "=" * 78)
print("D2.2: RL-only (reward=ROUGE-L) − MLE-only, генерация ответов ED")
print("=" * 78)
d = [r["RL-only"] - r["MLE-only"] for r in paired_d22]
n = len(d)
se = np.std(d, ddof=1) / sqrt(n) if n > 1 else float("nan")
t = np.mean(d) / se if se and se > 0 else float("nan")
tc = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365}[n]
per_seed_str = [f"s{r['seed']}: {r['RL-only'] - r['MLE-only']:+.4f}" for r in paired_d22]
print(f"per-seed: {per_seed_str}")
print(f"разность rougeL: {np.mean(d):+.4f} ± {np.std(d, ddof=1):.4f} | t={t:+.2f} "
      f"(крит {tc}) | {sum(x > 0 for x in d)}/{n} в плюс")

diffs = []
for r in paired_d22:
    pa = f"{cfg2_baseline.output_dir}_d21_seed{r['seed']}/results.json"
    pb = f"{cfg2_rl.output_dir}_d22_seed{r['seed']}/results.json"
    if os.path.exists(pa) and os.path.exists(pb):
        va = json.load(open(pa)).get("per_example_rougeL")
        vb = json.load(open(pb)).get("per_example_rougeL")
        if va and vb:
            diffs += [y - x for x, y in zip(va, vb)]
p = float("nan")
if diffs:
    arr = np.array(diffs, float)
    rng = np.random.default_rng(42)
    boots = np.array([arr[rng.integers(0, len(arr), len(arr))].mean()
                      for _ in range(10000)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    print(f"бутстреп: {arr.mean():+.4f} [CI {lo:+.4f}, {hi:+.4f}], "
          f"p={p:.4f}, n={len(arr)} пар")

print("\nDistinct-1: " + " | ".join(
    f"s{r['seed']}: MLE {r['MLE-only-d1']:.3f} vs RL {r['RL-only-d1']:.3f}"
    for r in paired_d22))

print("\nТраектория RL-награды по прогонам (первые/последние события):")
for r in paired_d22:
    pj = f"{cfg2_rl.output_dir}_d22_seed{r['seed']}/results.json"
    if os.path.exists(pj):
        hist = json.load(open(pj)).get("rl_reward_history") or []
        if hist:
            first5 = np.mean([h["reward"] for h in hist[:5]])
            last5 = np.mean([h["reward"] for h in hist[-5:]])
            print(f"  s{r['seed']}: событий {len(hist)} | "
                  f"reward {first5:.4f} (первые 5) -> {last5:.4f} (последние 5) | "
                  f"динамика {last5-first5:+.4f}")

print("\nВЕРДИКТ (протокол §4.16: t + бутстреп + репликация направления):")
t_ok = abs(t) > tc
p_ok = (not np.isnan(p)) and p < 0.05
if t_ok and p_ok:
    print(f"  RL-only > MLE ПОДТВЕРЖДЁН на n={n}.")
elif all(x > 0 for x in d):
    print(f"  Направление {sum(x > 0 for x in d)}/{n} в плюс, значимости нет "
          f"(t={t:+.2f} при {tc}, p={p:.3f}).")
elif all(x < 0 for x in d):
    print(f"  УСТОЙЧИВО ОТРИЦАТЕЛЬНО на {sum(x < 0 for x in d)}/{n} сидов "
          f"(t={t:+.2f}, p={p:.3f}) — RL мешает и в генерации.")
else:
    print(f"  Смешанно — эффекта нет (t={t:+.2f}, p={p:.3f}).")

with open(_summary_d22_path, "w") as f:
    json.dump({"rows": paired_d22,
               "pair_rl_mle": {"n": n, "mean": float(np.mean(d)),
                               "std": float(np.std(d, ddof=1)),
                               "t": float(t), "t_crit": tc,
                               "p_bootstrap": float(p)}}, f, indent=2)
print(f"\nИтог: {_summary_d22_path}")

D2.2: RL-only | reward=rougeL | событие каждые 100 шагов по 64 примерам
[Resume] сид 42 уже завершён
[Resume] сид 43 уже завершён
[Resume] сид 44 уже завершён
[Paired] сид 45: MLE D2.1 rougeL=0.1736, общий старт с диска

STARTING EXPERIMENT: RL-only D2.2 (seed 45, общий старт D2.1)



Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10497.79it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 4.291e-07
[RL sample] REF: 'I am sure you were happy'
[RL sample] GEN: "That's great! What did you do to get those A's?"
[RL step 0] reward=0.1486 adv=+0.1486 gamma=4.29e-07
[Step 0] CE/token (EMA): 2.547 | Contrast[mask]: 0.000 | RL: -0.764 | Reward: 0.1486


Epoch,Training Loss,Validation Loss
1,2.263601,2.328975
2,2.179647,2.314012
3,2.200630,2.312877
4,2.240124,2.313389


[Step 10] CE/token (EMA): 2.444 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.388 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.332 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.296 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.259 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.250 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.218 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.257 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.285 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: "What's wrong with your tooth? I hope it's not too painful of an experience."
[RL sample] GEN: "I'm sorry to hear that. I hope it doesn't hurt too much."
[RL step 100] reward=0.1758 adv=+0.1610 gamma=4.29e-07
[Step 100

...

[Step 10] CE/token (EMA): 2.095 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.108 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.155 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.197 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.199 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.239 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.205 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.203 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.244 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: "What's wrong with your tooth? I hope it's not too painful of an experience."
[RL sample] GEN: "I know what you mean. I hate going to the dentist. I'm not sure if I'll ever get over it."
[RL step 100] reward=0.1485 adv

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10449.76it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 3.989e-07
[RL sample] REF: 'I am sure you were happy'
[RL sample] GEN: "That's awesome! I'm sure you're proud of yourself."
[RL step 0] reward=0.1505 adv=+0.1505 gamma=3.99e-07
[Step 0] CE/token (EMA): 2.399 | Contrast[mask]: 0.000 | RL: -0.720 | Reward: 0.1505


Epoch,Training Loss,Validation Loss
1,2.339852,2.316792
2,2.099730,2.296912
3,2.147129,2.311697
4,2.219945,2.319461


[Step 10] CE/token (EMA): 2.328 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.328 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.259 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.296 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.233 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.198 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.187 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.220 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.228 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: "What's wrong with your tooth? I hope it's not too painful of an experience."
[RL sample] GEN: 'I know what you mean. I hate going to the dentist.'
[RL step 100] reward=0.1553 adv=+0.1402 gamma=3.99e-07
[Step 100] CE/t

In [ ]:
# ============================================================
# CELL D2-9: BERTScore по СОХРАННЁННЫМ predictions D2
# (требует: pip install bert-score; первый запуск скачает roberta-large)
# Считает per-example F1 (rescale_with_baseline) для всех конфигов
# D2.1 + D2.2 и сохраняет в bertscores_d2.json (для бутстрепа/отчёта).
# ============================================================
try:
    from bert_score import score as bert_score_fn
except ImportError:
    bert_score_fn = None
    print("bert-score НЕ установлен: pip install bert-score, затем перезапустить ячейку")

if bert_score_fn:
    BS_CONFIGS = [
        ("MLE-only", "./gen_d2_mle", "d21"),
        ("contrast-mask", "./gen_d2_mask", "d21"),
        ("contrast-offtopic", "./gen_d2_offtopic", "d21"),
        ("contrast-incoherent", "./gen_d2_incoherent", "d21"),
        ("RL-only", "./gen_d2_rl_only", "d22"),
    ]
    refs_bs = list(test2_references)
    device_bs = "cuda" if torch.cuda.is_available() else "cpu"
    bertscores = {}
    for name, bdir, wave in BS_CONFIGS:
        per_seed_f1 = []
        for seed in (42, 43, 44):
            p = f"{bdir}_{wave}_seed{seed}/results.json"
            if not os.path.exists(p):
                continue
            preds = json.load(open(p)).get("predictions")
            if not preds:
                continue
            _, _, f1_bs = bert_score_fn(preds, refs_bs, lang="en",
                                        rescale_with_baseline=True, verbose=False,
                                        batch_size=32, device=device_bs)
            per_seed_f1.append([float(x) for x in f1_bs])
        if per_seed_f1:
            bertscores[name] = per_seed_f1
            means = [float(np.mean(v)) for v in per_seed_f1]
            print(f"{name:<22} BERTScore-F1(rescaled): {np.mean(means):+.4f} "
                  f"± {np.std(means):.4f}  per-seed {[f'{m:+.4f}' for m in means]}")

    if "MLE-only" in bertscores:
        print("\nПарные разности к MLE (по сидам, mean):")
        for name in bertscores:
            if name == "MLE-only":
                continue
            dd = [np.mean(vb) - np.mean(va)
                  for va, vb in zip(bertscores["MLE-only"], bertscores[name])]
            if dd:
                print(f"  {name:<22} − MLE: {np.mean(dd):+.4f} "
                      f"{[f'{x:+.4f}' for x in dd]}")

    os.makedirs(cfg2.output_dir, exist_ok=True)
    with open(f"{cfg2.output_dir}/bertscores_d2.json", "w") as f:
        json.dump(bertscores, f)
    print(f"\nСохранено: {cfg2.output_dir}/bertscores_d2.json")
else:
    print("Пропуск BERTScore")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11026.90it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
<VENV>/lib/python3.14/site-packages/bert_score/score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may wan

MLE-only               BERTScore-F1(rescaled): +0.2833 ± 0.0032  per-seed ['+0.2862', '+0.2848', '+0.2789']


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10355.12it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11046.46it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UN

contrast-mask          BERTScore-F1(rescaled): +0.2912 ± 0.0080  per-seed ['+0.2870', '+0.2841', '+0.3024']


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10963.62it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10553.93it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UN

contrast-offtopic      BERTScore-F1(rescaled): +0.2815 ± 0.0027  per-seed ['+0.2797', '+0.2853', '+0.2795']


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10640.30it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11201.17it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UN

contrast-incoherent    BERTScore-F1(rescaled): +0.2495 ± 0.0654  per-seed ['+0.1570', '+0.2986', '+0.2928']


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10831.23it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11091.37it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UN

RL-only                BERTScore-F1(rescaled): +0.2805 ± 0.0069  per-seed ['+0.2832', '+0.2873', '+0.2710']

Парные разности к MLE (по сидам, mean):
  contrast-mask          − MLE: +0.0079 ['+0.0007', '-0.0006', '+0.0236']
  contrast-offtopic      − MLE: -0.0018 ['-0.0065', '+0.0006', '+0.0006']
  contrast-incoherent    − MLE: -0.0338 ['-0.1292', '+0.0138', '+0.0140']
  RL-only                − MLE: -0.0028 ['-0.0031', '+0.0025', '-0.0078']

Сохранено: ./gen_dialogue_ed_d2/bertscores_d2.json


In [ ]:
# ============================================================
# CELL D2-10: КАУЗАЛЬНЫЙ ТЕСТ incoherent s42 (§5.1)
# Вопрос: деградация s42 (rougeL 0.128, distinct-1 0.015, CE-плато
# 3.7-4.6 с шага ~800) — ПРИЧИНА incoherent-негатива или реализация
# хаоса? Дизайн: перезапуск с ТЕМ ЖЕ shared_init.pt (тот же сид 42):
#   1) КОНТРОЛЬ — MLE s42: детерминирован ли пайплайн D2 вообще
#      (в D1 MLE/mask воспроизводились бит-в-бит, wl — нет);
#   2) ЛЕЧЕНИЕ — incoherent s42: деградирует снова -> причинено
#      негативом; здоров -> стохастическое событие (негатив = фактор
#      риска, не детерминированная причина).
# Оригиналы не трогаем — новые папки _d23_*_{tag}. Время ~35 мин.
# ============================================================
import copy

SEED_CT = 42
_mle_dir_orig = f"{cfg2_baseline.output_dir}_d21_seed{SEED_CT}"
_inc_dir_orig = f"{cfg2_inc.output_dir}_d21_seed{SEED_CT}"
assert os.path.exists(os.path.join(_mle_dir_orig, "shared_init.pt")), \
    "нет общего старта D2.1 для сида 42"
shared_ct = torch.load(os.path.join(_mle_dir_orig, "shared_init.pt"),
                       map_location="cpu")


def _rerun(base_cfg, tag):
    """Перезапуск с общего старта; resume-safe; возвращает results.json."""
    c = copy.copy(base_cfg)
    c.seed = SEED_CT
    c.output_dir = f"{base_cfg.output_dir}_d23_seed{SEED_CT}_{tag}"
    rj = os.path.join(c.output_dir, "results.json")
    if not os.path.exists(rj):
        run_experiment(
            c, f"{tag} D2.3 каузальный тест (seed {SEED_CT}, старт D2.1)",
            init_state=shared_ct)
        gc.collect()
        torch.cuda.empty_cache()
    with open(rj) as f:
        return json.load(f)


res_mle_repro = _rerun(cfg2_baseline, "MLE-repro")
res_inc_repro = _rerun(cfg2_inc, "incoherent-repro")


def _summ(res, name):
    sc = res["test_scores"]
    hist = res.get("val_metric", {}).get("history", [])
    tr = " -> ".join(f"{h['value']:.3f}" for h in hist)
    print(f"  {name:<26} rougeL={sc['rougeL']:.4f}  d1={sc['distinct1']:.4f}"
          f"  | val: {tr}")
    return sc["rougeL"]


print("=" * 86)
print(f"КАУЗАЛЬНЫЙ ТЕСТ incoherent s{SEED_CT} (общий старт, тот же сид)")
print("=" * 86)
with open(f"{_mle_dir_orig}/results.json") as f:
    m_orig = _summ(json.load(f), "MLE оригинал (D2.1)")
m_rep = _summ(res_mle_repro, "MLE перезапуск")
with open(f"{_inc_dir_orig}/results.json") as f:
    i_orig = _summ(json.load(f), "incoherent оригинал")
i_rep = _summ(res_inc_repro, "incoherent перезапуск")

m_det = abs(m_rep - m_orig) < 1e-6
print(f"\n1) Контроль детерминизма (MLE): "
      f"{'ВОСПРОИЗВЁЛСЯ БИТ-В-БИТ' if m_det else 'РАЗОШЛИСЯ'} "
      f"(ориг {m_orig:.4f} vs репро {m_rep:.4f})")

THR = 0.16   # здоровые прогоны D2: 0.169-0.188; деградировавший: 0.128
i_deg = i_rep < THR
print(f"2) incoherent перезапуск: rougeL={i_rep:.4f} "
      f"({'ДЕГРАДИРОВАЛ СНОВА' if i_deg else 'ЗДОРОВ'}) (порог {THR})")

if m_det and i_deg:
    verdict = ("ПРИЧИНЕН: пайплайн детерминирован и деградация "
               "воспроизводится — виноват incoherent-негатив "
               "(отталкивание от собственных признаков позитива)")
elif m_det and not i_deg:
    verdict = ("ХАОС: пайплайн детерминирован, деградация НЕ "
               "воспроизвелась — стохастическое событие; негатив = "
               "фактор риска, не детерминированная причина")
elif not m_det and i_deg:
    verdict = ("тренд ПРИЧИНЕН: деградация воспроизвелась, но пайплайн "
               "недетерминирован — убедительно, не строго")
else:
    verdict = ("тренд ХАОС: пайплайн недетерминирован, деградация не "
               "воспроизвелась — трактовать как хаос")
print(f"3) ВЕРДИКТ: {verdict}")
print("=" * 86)

КАУЗАЛЬНЫЙ ТЕСТ incoherent s42 (общий старт, тот же сид)
  MLE оригинал (D2.1)        rougeL=0.1849  d1=0.1232  | val: 0.173 -> 0.157 -> 0.161 -> 0.168
  MLE перезапуск             rougeL=0.1849  d1=0.1232  | val: 0.173 -> 0.157 -> 0.161 -> 0.168
  incoherent оригинал        rougeL=0.1283  d1=0.0150  | val: 0.109 -> 0.081 -> 0.118 -> 0.115
  incoherent перезапуск      rougeL=0.1283  d1=0.0150  | val: 0.109 -> 0.081 -> 0.118 -> 0.115

1) Контроль детерминизма (MLE): ВОСПРОИЗВЁЛСЯ БИТ-В-БИТ (ориг 0.1849 vs репро 0.1849)
2) incoherent перезапуск: rougeL=0.1283 (ДЕГРАДИРОВАЛ СНОВА) (порог 0.16)
3) ВЕРДИКТ: ПРИЧИНЕН: пайплайн детерминирован и деградация воспроизводится — виноват incoherent-негатив (отталкивание от собственных признаков позитива)


In [ ]:
# ============================================================
# CELL D2-11a: хелпер контрастного лосса для диагностики —
# переиспользует _contrastive_loss тренера через НАСТОЯЩИЙ экземпляр,
# созданный через __new__ (без __init__ с TrainingArguments): все
# методы — static и обычные — резолвятся штатно, без ручного
# связывания (два предыдущих варианта прокси падали: несвязанный
# обычный метод -> missing 'labels'; .__get__ на staticmethod-
# функции -> лишний self)
# ============================================================
def _grad_trainer_contrast(model, mcfg, batch, h_pos, plist):
    """Контрастный лосс учебной конфигурации (негатив detached и т.д.)."""
    pr = TwoPhaseTrainerGen.__new__(TwoPhaseTrainerGen)  # без __init__
    pr.k_negatives = mcfg.k_negatives
    pr.contrast_mode = mcfg.contrast_mode
    pr.contrastive_dropout = mcfg.contrastive_dropout
    pr.contrastive_tau = mcfg.contrastive_tau
    pr.contrast_params = [(n, p) for n, p in plist
                          if "embedding" in n] or list(plist)
    return pr._contrastive_loss(model, batch, h_pos)

print("хелпер _grad_trainer_contrast готов (прокси = экземпляр через __new__)")

хелпер _grad_trainer_contrast готов (прокси = экземпляр через __new__)


In [ ]:
# ============================================================
# CELL D2-11: ГРАДИЕНТНАЯ ДИАГНОСТИКА конфликта CE vs контраст
# (ответ на критику §17/§47: механизм коллапса — «hypothesized»,
# пока не измерены sim-распределения и градиенты.)
# Без обучения: на ОБЩЕМ СТАРТЕ D2.1 (сид 42) по N батчам train:
#   E[sim(h+, h−)]  — распределение сходства позитив/негатив;
#   ||∇L_CE||, ||∇(β·L_ctr)|| (по параметрам префикса);
#   cos(∇L_CE, ∇(β·L_ctr)) — мера конфликта целей.
# Проверяемая гипотеза (§5.5): incoherent — конфликт (отрицательный
# cos при большой норме контраста); mask — малая норма/ортогонален;
# offtopic — промежуточно. Градиенты считаются в УЧЕБНОЙ
# конфигурации (негатив detached, как в тренировке).
# Время: ~5-10 мин (1 загрузка модели, 3 режима x N батчей).
# ============================================================
import torch.nn.functional as F
import json

GRAD_MODES = [("mask", cfg2_mask), ("offtopic", cfg2_off), ("incoherent", cfg2_inc)]
GRAD_SEED, GRAD_N_BATCHES = 42, 12

set_seed(GRAD_SEED)
_gc_base = AutoModelForCausalLM.from_pretrained(
    cfg2.model_name, dtype=torch.bfloat16, trust_remote_code=True)
_gc_model = get_peft_model(
    _gc_base,
    PrefixTuningConfig(task_type=TaskType.CAUSAL_LM,
                       num_virtual_tokens=cfg2.num_virtual_tokens,
                       prefix_projection=True, inference_mode=False))
for _p in _gc_model.parameters():
    if _p.requires_grad:
        _p.data = _p.data.float()
_gc_model.enable_input_require_grads()
_gc_model.to("cuda")

_shared_g = torch.load(f"{cfg2_baseline.output_dir}_d21_seed{GRAD_SEED}/shared_init.pt",
                       map_location="cpu")
_plist_g = [(n, p) for n, p in _gc_model.named_parameters()
            if p.requires_grad and "prompt_encoder" in n]
with torch.no_grad():
    for _n, _p in _plist_g:
        _p.data.copy_(_shared_g[_n].to(_p.device))
print(f"Общий старт D2.1 s{GRAD_SEED} загружен ({len(_plist_g)} тензоров)")

_grad_loader = DataLoader(train2_ds, batch_size=cfg2.batch_size,
                          shuffle=False, collate_fn=causal_lm_collator)


def _pool_g(hidden, labels):
    if hidden.size(1) != labels.size(1):
        hidden = hidden[:, -labels.size(1):, :]
    m = (labels != -100).unsqueeze(-1).to(hidden.dtype)
    return F.normalize(((hidden * m).sum(1) / m.sum(1).clamp(min=1)).float(), dim=-1)


grad_report = {}
for _mode, _mcfg in GRAD_MODES:
    sims, gces, gctrs, coses = [], [], [], []
    _gc_model.train()
    _it = iter(_grad_loader)
    for _bi in range(GRAD_N_BATCHES):
        _b = next(_it)
        _b = {k: v.to(_gc_model.device) for k, v in _b.items()}
        _mi = {k: _b[k] for k in ("input_ids", "attention_mask", "labels")}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            _out = _gc_model(**_mi, output_hidden_states=True)
            _loss_ce = _out.loss
            _h_pos = _pool_g(_out.hidden_states[-1], _mi["labels"])

            # sim(h+, h-) по режиму (для mask — реплика маскирования)
            if _mode == "mask":
                _cp = [(n, p) for n, p in _plist_g if "embedding" in n] or _plist_g
                _orig = {n: p.data.clone() for n, p in _cp}
                try:
                    for _n, _p in _cp:
                        _mk = (torch.rand_like(_p.float()) > _mcfg.contrastive_dropout).to(_p.dtype)
                        _p.data.copy_((_orig[_n] * _mk).to(_p.dtype))
                    with torch.no_grad():
                        _on = _gc_model(input_ids=_mi["input_ids"],
                                        attention_mask=_mi["attention_mask"],
                                        output_hidden_states=True)
                    _h_neg = _pool_g(_on.hidden_states[-1], _mi["labels"])
                finally:
                    for _n, _p in _cp:
                        _p.data.copy_(_orig[_n])
                del _on
            else:
                _pfx = {"offtopic": "negoff", "incoherent": "neginc"}[_mode]
                with torch.no_grad():
                    _on = _gc_model(input_ids=_b[f"{_pfx}_input_ids"],
                                    attention_mask=_b[f"{_pfx}_attention_mask"],
                                    output_hidden_states=True)
                _h_neg = _pool_g(_on.hidden_states[-1], _b[f"{_pfx}_labels"])
                del _on
            sims += torch.einsum("bd,bd->b", _h_pos, _h_neg.detach()).tolist()

            # градиенты учебного лосса: негатив detached (как в тренировке)
            _params = [p for _, p in _plist_g]
            _g_ce = torch.autograd.grad(_loss_ce, _params,
                                        retain_graph=True, allow_unused=True)
            _loss_ctr = _grad_trainer_contrast(_gc_model, _mcfg, _b, _h_pos, _plist_g)
            _g_ct = torch.autograd.grad(_mcfg.beta * _loss_ctr, _params,
                                        allow_unused=True)
            _gce = torch.cat([g.reshape(-1) for g in _g_ce if g is not None])
            _gct = torch.cat([g.reshape(-1) for g in _g_ct if g is not None])
            gces.append(float(_gce.norm()))
            gctrs.append(float(_gct.norm()))
            coses.append(float(F.cosine_similarity(_gce, _gct, dim=0)))
    grad_report[_mode] = {
        "sim_mean": float(np.mean(sims)), "sim_std": float(np.std(sims)),
        "g_ce": float(np.mean(gces)), "g_ctr": float(np.mean(gctrs)),
        "g_ctr_ratio": float(np.mean(gctrs) / np.mean(gces)),
        "cos_mean": float(np.mean(coses)), "cos_std": float(np.std(coses)),
    }
    del _it

print("\n" + "=" * 92)
print(f"ГРАДИЕНТНАЯ ДИАГНОСТИКА на общем старте D2.1 (s{GRAD_SEED}), "
      f"{GRAD_N_BATCHES} батчей, bf16-autocast, негатив detached")
print("=" * 92)
print(f"{'Режим':<14}{'E[sim(h+,h−)]':<15}{'||g_CE||':<11}{'||βg_ctr||':<12}"
      f"{'норма/CE':<10}{'cos(g_CE,g_ctr)':<18}")
for _mode, r in grad_report.items():
    print(f"{_mode:<14}{r['sim_mean']:<15.4f}{r['g_ce']:<11.4f}{r['g_ctr']:<12.4f}"
          f"{r['g_ctr_ratio']:<10.3f}{r['cos_mean']:+.4f} ± {r['cos_std']:.4f}")
print("=" * 92)
print("Интерпретация: конфликт целей = отрицательный cos при большой")
print("норме контраста; вырожденная оценка = одна большая норма без")
print("конфликта направлений. Сравнить с исходом обучения (§5.1/§5.5).")

os.makedirs(cfg2.output_dir, exist_ok=True)
with open(f"{cfg2.output_dir}/grad_diagnostics.json", "w") as f:
    json.dump(grad_report, f, indent=2)
print(f"Сохранено: {cfg2.output_dir}/grad_diagnostics.json")

del _gc_model, _gc_base
gc.collect()
torch.cuda.empty_cache()

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9114.31it/s]


Общий старт D2.1 s42 загружен (5 тензоров)

ГРАДИЕНТНАЯ ДИАГНОСТИКА на общем старте D2.1 (s42), 12 батчей, bf16-autocast, негатив detached
Режим         E[sim(h+,h−)]  ||g_CE||   ||βg_ctr||  норма/CE  cos(g_CE,g_ctr)   
mask          0.9999         0.4562     0.0011      0.002     -0.0122 ± 0.0807
offtopic      0.9626         0.4562     0.0024      0.005     +0.1024 ± 0.0697
incoherent    0.9932         0.4562     0.0012      0.003     +0.1144 ± 0.1015
Интерпретация: конфликт целей = отрицательный cos при большой
норме контраста; вырожденная оценка = одна большая норма без
конфликта направлений. Сравнить с исходом обучения (§5.1/§5.5).
Сохранено: ./gen_dialogue_ed_d2/grad_diagnostics.json


In [ ]:
# ============================================================
# CELL D2-12: АБЛЯЦИИ D2.3 (критика-2): 3 сида x 2 конфига
#   1) offtopic-grad — градиент и через негатив (симметричный
#      апдейт; учебная конфигурация D2.1 — detached);
#   2) offtopic4 — 4 различных offtopic-негатива (k=1 -> k=4,
#      как Σ_j в Eq. 3 статьи).
# Пары с MLE D2.1 (shared_init.pt с диска), сиды 42-44.
# Время: offtopic4 ~38 мин/сид, offtopic-grad ~28 мин/сид -> ~3.3 ч.
# ВНИМАНИЕ (offtopic-grad): активации двух графов; при OOM —
# batch 4 + accum 2 в конфиге.
# ============================================================
import copy
from math import sqrt


class D2_Offtopic_Grad(CFG_D2):
    """Симметричный контраст: негатив в графе вычислений."""
    output_dir = "./gen_d2_offtopic_grad"

    beta = 0.1
    contrast_mode = "offtopic_grad"
    contrast_neg_in_graph = True


class D2_Offtopic_K4(CFG_D2):
    """4 offtopic-негатива на пример (k=4 вместо 1)."""
    output_dir = "./gen_d2_offtopic4"

    beta = 0.1
    contrast_mode = "offtopic4"


ABLT_CONFIGS = [
    (D2_Offtopic_K4(), "offtopic4"),        # сначала k=4 (без риска OOM)
    (D2_Offtopic_Grad(), "offtopic-grad"),
]
ABLT_SEEDS = [42, 43, 44]

_summary_abl = f"{cfg2.output_dir}/summary_d2_3.json"
abl_rows = []
if os.path.exists(_summary_abl):
    with open(_summary_abl) as f:
        abl_rows = json.load(f).get("rows", [])

for base_cfg, name in ABLT_CONFIGS:
    for seed in ABLT_SEEDS:
        if any(r.get("seed") == seed and r.get("config") == name
               for r in abl_rows):
            print(f"[Resume] {name} s{seed} уже завершён")
            continue
        mle_dir = f"{cfg2_baseline.output_dir}_d21_seed{seed}"
        assert os.path.exists(os.path.join(mle_dir, "results.json")), \
            f"нет MLE D2.1 для сида {seed}"
        shared = torch.load(os.path.join(mle_dir, "shared_init.pt"),
                            map_location="cpu")
        with open(os.path.join(mle_dir, "results.json")) as f:
            mle_scores = json.load(f)["test_scores"]
        c = copy.copy(base_cfg)
        c.seed = seed
        c.output_dir = f"{base_cfg.output_dir}_d23_seed{seed}"
        s = run_experiment(c, f"{name} D2.3 (seed {seed}, старт D2.1)",
                           init_state=shared)
        abl_rows = [r for r in abl_rows
                    if not (r.get("seed") == seed and r.get("config") == name)]
        abl_rows.append({"config": name, "seed": seed,
                         "MLE-only": mle_scores["rougeL"],
                         "rougeL": s["rougeL"],
                         "distinct1": s["distinct1"]})
        del shared
        gc.collect()
        torch.cuda.empty_cache()
        with open(_summary_abl, "w") as f:
            json.dump({"rows": abl_rows}, f, indent=2)
        print(f">>> {name} s{seed}: rougeL={s['rougeL']:.4f} "
              f"(MLE {mle_scores['rougeL']:.4f}, "
              f"diff {s['rougeL']-mle_scores['rougeL']:+.4f})\n")

# ---------- агрегация ----------
print("\n" + "=" * 88)
print("D2.3: абляции негативов (3 сида, общий старт с D2.1)")
print("=" * 88)
off_d21 = {r["seed"]: r["contrast-offtopic"] for r in paired_d21}
mask_d21 = {r["seed"]: r["contrast-mask"] for r in paired_d21}
for name in ("offtopic4", "offtopic-grad"):
    rows_n = [r for r in abl_rows if r["config"] == name]
    if not rows_n:
        continue
    d = [r["rougeL"] - r["MLE-only"] for r in rows_n]
    n = len(d)
    se = np.std(d, ddof=1) / sqrt(n) if n > 1 else float("nan")
    t = np.mean(d) / se if se and se > 0 else float("nan")
    print(f"\n{name} − MLE: {np.mean(d):+.4f} ± {np.std(d, ddof=1):.4f} "
          f"| t={t:+.2f} | per-seed {[f'{x:+.4f}' for x in d]}")
    cmp_off = [r["rougeL"] - off_d21[r["seed"]]
               for r in rows_n if r["seed"] in off_d21]
    cmp_mask = [r["rougeL"] - mask_d21[r["seed"]]
                for r in rows_n if r["seed"] in mask_d21]
    if cmp_off:
        print(f"  vs offtopic(D2.1, detached k=1): {np.mean(cmp_off):+.4f} "
              f"{[f'{x:+.4f}' for x in cmp_off]}")
    if cmp_mask:
        print(f"  vs mask(D2.1):                  {np.mean(cmp_mask):+.4f} "
              f"{[f'{x:+.4f}' for x in cmp_mask]}")
print("\nВопросы: (1) меняет ли симметричный апдейт знак/величину "
      "эффекта offtopic? (2) дают ли 4 негатива усиление?")

with open(_summary_abl, "w") as f:
    json.dump({"rows": abl_rows}, f, indent=2)
print(f"Итог: {_summary_abl}")


STARTING EXPERIMENT: offtopic4 D2.3 (seed 42, старт D2.1)



Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10020.91it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=offtopic4, from_start=True, beta=0.1
[Step 0] CE/token (EMA): 1.981 | Contrast[offtopic4]: 1.544 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,2.414435,2.477104
2,2.539450,2.467068
3,2.453241,2.463049
4,2.311723,2.473469


[Step 10] CE/token (EMA): 2.076 | Contrast[offtopic4]: 1.552 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.122 | Contrast[offtopic4]: 1.553 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.169 | Contrast[offtopic4]: 1.554 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.263 | Contrast[offtopic4]: 1.540 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.229 | Contrast[offtopic4]: 1.558 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.257 | Contrast[offtopic4]: 1.551 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.236 | Contrast[offtopic4]: 1.538 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.254 | Contrast[offtopic4]: 1.555 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.249 | Contrast[offtopic4]: 1.558 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 2.274 | Contrast[offtopic4]: 1.544 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 2.281 | Contrast[offtopic4]: 1.544 | RL: 0.000 | Reward: 0.0000
[Step 12

...

[Step 10] CE/token (EMA): 2.291 | Contrast[offtopic4]: 1.546 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.265 | Contrast[offtopic4]: 1.549 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.234 | Contrast[offtopic4]: 1.547 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.243 | Contrast[offtopic4]: 1.551 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.197 | Contrast[offtopic4]: 1.548 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.157 | Contrast[offtopic4]: 1.553 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.234 | Contrast[offtopic4]: 1.554 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.213 | Contrast[offtopic4]: 1.548 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.280 | Contrast[offtopic4]: 1.540 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 2.291 | Contrast[offtopic4]: 1.544 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 2.294 | Contrast[offtopic4]: 1.543 | RL: 0.000 | Reward: 0.0000
[Step 12

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9818.01it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=offtopic_grad, from_start=True, beta=0.1
[Step 0] CE/token (EMA): 1.981 | Contrast[offtopic_grad]: 0.654 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 484.00 MiB. GPU 0 has a total capacity of 15.48 GiB of which 381.44 MiB is free. Process 108287 has 80.41 MiB memory in use. Process 120695 has 238.00 MiB memory in use. Process 2816 has 4.47 MiB memory in use. Process 166670 has 28.72 MiB memory in use. Including non-PyTorch memory, this process has 14.38 GiB memory in use. Of the allocated memory 13.98 GiB is allocated by PyTorch, and 229.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# ============================================================
# CELL D2-13 (ОПЦИЯ, ~1 ч): RL-вариант по критике-2 —
# сэмплированная награда (do_sample, фикс. температура) вместо
# жадной + быстрый бейзлайн (EMA 0.5 вместо 0.9).
# Гипотезы критика: жадная награда не дифференцирует похожие
# промпты; медленный бейзлайн даёт стабильно положительный
# advantage без контраста хороший/плохой.
# Пары с MLE D2.1, сиды 42-44, итог summary_d2_2s.json.
# ============================================================
import copy
from math import sqrt


class D2_RL_Sample(CFG_D2):
    output_dir = "./gen_d2_rl_sample"

    beta = 0.0
    contrast_mode = "mask"          # не используется при β=0
    gamma = 2e-5                    # старто́вое; auto перепишет
    gamma_auto = True
    rl_interval = 100
    rl_num_batches = 8
    gen_do_sample = True            # сэмплированная награда
    gen_temperature = 0.7
    gen_top_p = 0.9
    reward_baseline_beta = 0.5      # быстрый бейзлайн


cfg2_rl_s = D2_RL_Sample()
_summary_rls = f"{cfg2.output_dir}/summary_d2_2s.json"
rls_rows = []
if os.path.exists(_summary_rls):
    with open(_summary_rls) as f:
        rls_rows = json.load(f).get("rows", [])

for seed in (42, 43, 44):
    if any(r["seed"] == seed for r in rls_rows):
        print(f"[Resume] RL-sample s{seed} уже завершён")
        continue
    mle_dir = f"{cfg2_baseline.output_dir}_d21_seed{seed}"
    assert os.path.exists(os.path.join(mle_dir, "results.json"))
    shared = torch.load(os.path.join(mle_dir, "shared_init.pt"),
                        map_location="cpu")
    with open(os.path.join(mle_dir, "results.json")) as f:
        mle_sc = json.load(f)["test_scores"]
    c = copy.copy(cfg2_rl_s)
    c.seed = seed
    c.output_dir = f"{cfg2_rl_s.output_dir}_d22s_seed{seed}"
    sc = run_experiment(c, f"RL-sample D2.2b (seed {seed}, старт D2.1)",
                        init_state=shared)
    rls_rows = [r for r in rls_rows if r["seed"] != seed] + [
        {"seed": seed, "MLE-only": mle_sc["rougeL"], "RL-sample": sc["rougeL"],
         "distinct1": sc["distinct1"]}]
    del shared
    gc.collect()
    torch.cuda.empty_cache()
    with open(_summary_rls, "w") as f:
        json.dump({"rows": rls_rows}, f, indent=2)

d = [r["RL-sample"] - r["MLE-only"] for r in rls_rows]
n = len(d)
se = np.std(d, ddof=1) / sqrt(n) if n > 1 else float("nan")
t = np.mean(d) / se if se and se > 0 else float("nan")
print("\n" + "=" * 80)
print("D2.2b: RL-sample (сэмпл. награда + бейзлайн 0.5) − MLE, n=3")
print("=" * 80)
print(f"per-seed {[f'{x:+.4f}' for x in d]} | mean {np.mean(d):+.4f} "
      f"± {np.std(d, ddof=1):.4f} | t={t:+.2f}")
print("Сравнить с D2.2 (жадная награда, бейзлайн 0.9): diff был +0.0000.")


STARTING EXPERIMENT: RL-sample D2.2b (seed 42, старт D2.1)



Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9888.73it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 4.038e-07
[RL sample] REF: 'I am sure you were happy'
[RL sample] GEN: 'That is awesome. I hope you are continuing to do well in college.'
[RL step 0] reward=0.1228 adv=+0.1228 gamma=4.04e-07
[Step 0] CE/token (EMA): 1.981 | Contrast[mask]: 0.000 | RL: -0.594 | Reward: 0.1228


Epoch,Training Loss,Validation Loss
1,2.255201,2.319763
2,2.380507,2.304765
3,2.302304,2.310538
4,2.165400,2.325841


[Step 10] CE/token (EMA): 2.076 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.122 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.167 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.261 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.227 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.257 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.237 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.254 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.248 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: "What's wrong with your tooth? I hope it's not too painful of an experience."
[RL sample] GEN: "I hate the dentist too, but I have to go every couple of years.  I'm going to have to get one soon."
[RL step 100] reward=

...

[Step 10] CE/token (EMA): 2.217 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.233 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.254 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.236 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.262 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.305 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.247 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.188 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.243 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: "What's wrong with your tooth? I hope it's not too painful of an experience."
[RL sample] GEN: "That's not fun. What are you worried about?"
[RL step 100] reward=0.1322 adv=+0.0752 gamma=4.73e-07
[Step 100] CE/token (E

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9778.88it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 4.105e-07
[RL sample] REF: 'I am sure you were happy'
[RL sample] GEN: "That's great! What did you major in?"
[RL step 0] reward=0.1395 adv=+0.1395 gamma=4.10e-07
[Step 0] CE/token (EMA): 2.288 | Contrast[mask]: 0.000 | RL: -0.686 | Reward: 0.1395


Epoch,Training Loss,Validation Loss
1,2.362468,2.303300
2,2.242201,2.315982
3,2.139096,2.312567
4,2.165913,2.315096


[Step 10] CE/token (EMA): 2.291 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.264 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.234 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.242 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.197 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.158 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.234 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.213 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.281 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: "What's wrong with your tooth? I hope it's not too painful of an experience."
[RL sample] GEN: "I hate going to the dentist too. But it's necessary. I think it's really important to take care of your teeth."
[RL step 1

In [ ]:
# ============================================================
# CELL D1-15: ПРОВЕРКА ЧИСЛОВОЙ ЭКВИВАЛЕНТНОСТИ текущего кода
# завершённым прогонам (после отката паддинга, журнал 6.3).
# Перезапускаем MLE D1.6 s42 с сохранённого общего старта в ОТДЕЛЬНУЮ
# папку и сравниваем с сохранённым results.json. Ожидание: БИТ-В-БИТ
# (прецеденты: перезапуск D1.6 MLE/mask; каузальный тест D2).
# Если совпало -> смешанная таблица 7 сидов легитимна; если НЕТ ->
# СТОП до разбора. Время ~13 мин. Выполнять при свободном GPU.
# ============================================================
import copy

# D2-блок данных перепривязывал глобальные имена к D2 — возвращаем D1
# (функции/объекты D1-ячейки данных всё ещё в памяти ядра)
_cfg_d1 = CFG()
assert _cfg_d1.task_kind == "emotion_cls"
_raw_rl_    = HFDataset.from_list(ed_train_full[:_cfg_d1.rl_subset_size])
_raw_train_ = HFDataset.from_list(ed_train_full[_cfg_d1.rl_subset_size:
                                                _cfg_d1.rl_subset_size + _cfg_d1.train_size])
_raw_val_   = HFDataset.from_list(ed_val_full[:_cfg_d1.val_size])
_raw_test_  = HFDataset.from_list(ed_test_full[:_cfg_d1.test_size])
train_ds = _raw_train_.map(tokenize_gen, batched=True, remove_columns=_raw_train_.column_names)
val_ds   = _raw_val_.map(tokenize_gen, batched=True, remove_columns=_raw_val_.column_names)
test_ds  = _raw_test_.map(tokenize_gen, batched=True, remove_columns=_raw_test_.column_names)
rl_ds    = _raw_rl_.map(tokenize_gen, batched=True, remove_columns=_raw_rl_.column_names)
rl_ds = rl_ds.add_column("ref_idx", list(range(len(rl_ds))))
rl_references = _raw_rl_[_cfg_d1.target_column]
val_references = _raw_val_[_cfg_d1.target_column]
test_references = _raw_test_[_cfg_d1.target_column]
for _d in (train_ds, val_ds, test_ds, rl_ds):
    _d.set_format("torch")
print(f"D1-данные перепривязаны: train {len(train_ds)} | test {len(test_ds)}")

_mle_orig_dir = "./gen_crl_mle_baseline_d16_seed42"
with open(os.path.join(_mle_orig_dir, "results.json")) as f:
    _orig = json.load(f)
_shared_eq = torch.load(os.path.join(_mle_orig_dir, "shared_init.pt"),
                        map_location="cpu")

_c_eq = copy.copy(cfg_baseline)
_c_eq.seed = 42
_c_eq.output_dir = "./gen_crl_mle_baseline_d16_seed42_equivcheck"
_sc = run_experiment(_c_eq, "EQUIV-CHECK MLE D1.6 (s42, текущий код)",
                     init_state=_shared_eq)

print("\n" + "=" * 76)
print("ПРОВЕРКА ЭКВИВАЛЕНТНОСТИ: MLE D1.6 s42 (оригинал vs текущий код)")
print("=" * 76)
print(f"  test accuracy : ориг {_orig['test_scores']['accuracy']:.6f} | "
      f"новый {_sc['accuracy']:.6f}")
print(f"  verb accuracy : ориг {_orig['test_scores']['verb_accuracy']:.6f} | "
      f"новый {_sc['verb_accuracy']:.6f}")
_h_new = json.load(open(os.path.join(_c_eq.output_dir, "val_metric_history.json")))
_h_old = _orig["val_metric"]
same_hist = ([h["value"] for h in _h_new["history"]] ==
             [h["value"] for h in _h_old["history"]])
print(f"  val-траектория: {'ИДЕНТИЧНА' if same_hist else 'РАЗЛИЧАЕТСЯ'} "
      f"(ориг {[h['value'] for h in _h_old['history']]} | "
      f"новый {[h['value'] for h in _h_new['history']]})")
same_all = (abs(_sc["accuracy"] - _orig["test_scores"]["accuracy"]) < 1e-9
            and same_hist)
if same_all:
    print("\n  ВЕРДИКТ: БИТ-В-БИТ — текущий код эквивалентен завершённым "
          "прогонам; таблица 7 сидов легитимна.")
else:
    print("\n  ВЕРДИКТ: РАСХОЖДЕНИЕ — НЕ запускать ночной прогон до разбора!")
print("=" * 76)

del _shared_eq
gc.collect()
torch.cuda.empty_cache()

Map: 100%|██████████| 600/600 [00:00<00:00, 8632.65 examples/s]


D1-данные перепривязаны: train 6000 | test 500

STARTING EXPERIMENT: EQUIV-CHECK MLE D1.6 (s42, текущий код)



Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10675.72it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[Step 0] CE/token (EMA): 0.927 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,1.148999,1.172157
2,1.130557,1.124622
3,0.788621,1.079415
4,0.793256,1.080597


[Step 10] CE/token (EMA): 1.092 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 1.222 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 1.303 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 1.390 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 1.336 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 1.312 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 1.406 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 1.266 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 1.252 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 1.345 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 1.391 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 120] CE/token (EMA): 1.297 | Contrast[mask]: 0.000 | RL: 

In [14]:
# ============================================================
# CELL D1-16: ПРОДЛЕНИЕ RL-пары до n=10 (сиды 49, 50, 51)
# ПРЕДЗАЯВЛЕННОЕ правило (журнал §6.5): из-за пограничного результата
# RL−MLE на большом тесте (7/7 в плюс, t=2.51>2.447, бутстреп
# p=0.061, CI касается нуля) пара {MLE, RL-only} расширяется до
# РОВНО 10 сидов; все сиды включаются без отбора; дальнейших
# расширений НЕ будет (фиксация против optional stopping).
# Время: 3 × (MLE ~10 + RL ~18 + старт) ≈ 1.5 ч. После — перезапуск
# eval_bigtest.ipynb (возьмёт новые сиды и пересчитает вердикт).
# ============================================================
import copy

EXT_SEEDS = [49, 50, 51]

for seed in EXT_SEEDS:
    c_mle = copy.copy(cfg_baseline)
    c_mle.seed = seed
    c_mle.output_dir = f"{cfg_baseline.output_dir}_d16_seed{seed}"
    shared_path = os.path.join(c_mle.output_dir, "shared_init.pt")
    _mle_json = os.path.join(c_mle.output_dir, "results.json")
    if os.path.exists(shared_path) and os.path.exists(_mle_json):
        shared = torch.load(shared_path, map_location="cpu")
        with open(_mle_json) as f:
            scores_mle = json.load(f)["test_scores"]
        print(f"[Resume] сид {seed}: MLE с диска")
    else:
        scores_mle, shared = run_experiment(
            c_mle, f"MLE-only D1.6 (seed {seed})", return_init_state=True)
        os.makedirs(c_mle.output_dir, exist_ok=True)
        torch.save(shared, shared_path)
    assert shared is not None

    c_rl = copy.copy(cfg_rl)
    c_rl.seed = seed
    c_rl.output_dir = f"{cfg_rl.output_dir}_d16_seed{seed}"
    _rl_json = os.path.join(c_rl.output_dir, "results.json")
    if os.path.exists(_rl_json):
        with open(_rl_json) as f:
            scores_rl = json.load(f)["test_scores"]
        print(f"[Resume] сид {seed}: RL-only с диска")
    else:
        scores_rl = run_experiment(
            c_rl, f"RL-only D1.6 (seed {seed}, общий старт)", init_state=shared)

    print(f">>> сид {seed}: MLE={scores_mle['accuracy']:.4f} "
          f"RL={scores_rl['accuracy']:.4f} "
          f"(diff {scores_rl['accuracy']-scores_mle['accuracy']:+.4f})\n")
    del shared
    gc.collect()
    torch.cuda.empty_cache()

print("Продление завершено: пара {MLE, RL-only} = сиды 42–51 (n=10).")
print("Далее: перезапустить eval_bigtest.ipynb (D1-часть) — он подхватит")
print("новые прогоны и пересчитает t/бутстреп/вердикт на полном тесте.")


STARTING EXPERIMENT: MLE-only D1.6 (seed 49)



Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10230.76it/s]



[InitSel] best-of-4 | probe 200 опт. шагов | val CE на 200 примерах | cuda:0
[InitSel] кандидат 0: val CE = 1.4315
[InitSel] кандидат 1: val CE = 1.3474
[InitSel] кандидат 2: val CE = 1.3984
[InitSel] кандидат 3: val CE = 1.3687
[InitSel] выбран кандидат 1 (val CE 1.3474); ранжирование: #1: 1.347 | #3: 1.369 | #2: 1.398 | #0: 1.431

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[Step 0] CE/token (EMA): 1.074 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000


Epoch,Training Loss,Validation Loss
1,1.194007,1.240320
2,1.104635,1.186400
3,1.258035,1.071258
4,0.905622,1.106777


[Step 10] CE/token (EMA): 1.243 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 1.355 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 1.383 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 1.400 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 1.336 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 1.258 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 1.300 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 1.304 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 1.285 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 1.288 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 1.181 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 120] CE/token (EMA): 1.339 | Contrast[mask]: 0.000 | RL: 

...

[Step 10] CE/token (EMA): 1.415 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 1.486 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 1.515 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 1.557 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 1.297 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 1.245 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 1.346 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 1.256 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 1.251 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 100] CE/token (EMA): 1.211 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 110] CE/token (EMA): 1.217 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 120] CE/token (EMA): 1.301 | Contrast[mask]: 0.000 | RL: 

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9262.72it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[RL] gamma auto-calibrated to 5.985e-08
[RL sample] REF: 'prepared'
[RL sample] GEN: 'anxious\nAnticipating the'
[RL step 0] reward=0.4766 adv=+0.4766 gamma=5.99e-08
[Step 0] CE/token (EMA): 1.140 | Contrast[mask]: 0.000 | RL: -0.342 | Reward: 0.4766


Epoch,Training Loss,Validation Loss
1,1.539979,1.171498
2,1.200063,1.136926
3,1.332413,1.120582
4,1.152235,1.116036


[Step 10] CE/token (EMA): 1.414 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 1.486 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 1.513 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 1.552 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'content'
[RL sample] GEN: 'content\n\n### Explanation:\n\n*'
[RL step 50] reward=0.5625 adv=+0.5148 gamma=5.99e-08
[Step 50] CE/token (EMA): 1.296 | Contrast[mask]: 0.000 | RL: -0.369 | Reward: 0.5625
[Step 60] CE/token (EMA): 1.247 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 1.345 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 1.258 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 1.256 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[RL sample] REF: 'disappointed'
[RL sample] GEN: 'disappointed\n\nActions: I cried'
[RL step 100] rew